# 06 — Exploração e Tratamento do MapBiomas Cobertura e Uso da Terra

## Projeto AgroESG — Soja | Centro-Oeste e Sul | 2019–2024

Este notebook realiza a exploração, validação, tratamento e preparação dos
dados de cobertura e uso da terra do MapBiomas.

### Fonte

MapBiomas Brasil — Coleção 10.1  
Estatísticas de Cobertura e Uso da Terra por Municípios, Estados e Biomas.

Arquivo RAW:

`MAPBIOMAS_BRAZIL-COVERAGE_STATISTICS-COL.10.1-MUNICIPALITIES_STATES_BIOMES.xlsx`

### Escopo do projeto

**Cultura agrícola de referência:**
- Soja

**Regiões:**
- Centro-Oeste
- Sul

**UFs:**
- DF
- GO
- MS
- MT
- PR
- RS
- SC

**Período:**
- 2019 a 2024

### Objetivos deste notebook

1. Auditar a estrutura do arquivo RAW do MapBiomas.
2. Identificar as abas e metadados disponíveis.
3. Compreender os níveis de classificação de cobertura e uso da terra.
4. Recortar os dados para as UFs e anos do projeto.
5. Associar os municípios ao código IBGE.
6. Construir indicadores municipais de cobertura e uso da terra.
7. Gerar bases PROCESSED e CURATED para integrações posteriores.

> Os dados RAW serão mantidos sem alterações. Todas as transformações serão
> realizadas nas camadas PROCESSED e CURATED.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

pd.set_option(
    "display.max_columns",
    None
)

pd.set_option(
    "display.max_rows",
    100
)

print(
    "Pandas:",
    pd.__version__
)

print(
    "NumPy:",
    np.__version__
)

Pandas: 2.3.3
NumPy: 2.3.5


In [2]:
BASE_DIR = Path(
    r"C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao"
)


RAW_MAPBIOMAS_COBERTURA_DIR = (
    BASE_DIR
    / "data"
    / "raw"
    / "mapbiomas_cobertura"
)


PROCESSED_MAPBIOMAS_COBERTURA_DIR = (
    BASE_DIR
    / "data"
    / "databases_processed"
    / "mapbiomas_cobertura"
)


CURATED_MAPBIOMAS_COBERTURA_DIR = (
    BASE_DIR
    / "data"
    / "databases_curated"
    / "mapbiomas_cobertura"
)


QUALITY_MAPBIOMAS_COBERTURA_DIR = (
    PROCESSED_MAPBIOMAS_COBERTURA_DIR
    / "quality"
)


print(
    "RAW:"
)

print(
    RAW_MAPBIOMAS_COBERTURA_DIR
)


print(
    "\nPROCESSED:"
)

print(
    PROCESSED_MAPBIOMAS_COBERTURA_DIR
)


print(
    "\nCURATED:"
)

print(
    CURATED_MAPBIOMAS_COBERTURA_DIR
)

RAW:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\raw\mapbiomas_cobertura

PROCESSED:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\databases_processed\mapbiomas_cobertura

CURATED:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\databases_curated\mapbiomas_cobertura


In [3]:
PROCESSED_MAPBIOMAS_COBERTURA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CURATED_MAPBIOMAS_COBERTURA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

QUALITY_MAPBIOMAS_COBERTURA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "Pastas de saída preparadas."
)

Pastas de saída preparadas.


In [4]:
arquivos_excel = sorted(
    RAW_MAPBIOMAS_COBERTURA_DIR.glob(
        "*.xlsx"
    )
)

print(
    "Arquivos Excel encontrados:"
)

print(
    len(
        arquivos_excel
    )
)

for arquivo in arquivos_excel:

    tamanho_mb = (
        arquivo.stat().st_size
        / (1024 ** 2)
    )

    print(
        f"- {arquivo.name} "
        f"({tamanho_mb:.2f} MB)"
    )

Arquivos Excel encontrados:
1
- MAPBIOMAS_BRAZIL-COVERAGE_STATISTICS-COL.10.1-MUNICIPALITIES_STATES_BIOMES.xlsx (75.05 MB)


In [5]:
if len(arquivos_excel) != 1:

    raise ValueError(
        "Era esperado exatamente 1 arquivo Excel "
        f"na pasta RAW, mas foram encontrados "
        f"{len(arquivos_excel)}."
    )


ARQUIVO_MAPBIOMAS = (
    arquivos_excel[0]
)


print(
    "Arquivo RAW selecionado:"
)

print(
    ARQUIVO_MAPBIOMAS
)

Arquivo RAW selecionado:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\raw\mapbiomas_cobertura\MAPBIOMAS_BRAZIL-COVERAGE_STATISTICS-COL.10.1-MUNICIPALITIES_STATES_BIOMES.xlsx


In [6]:
excel_mapbiomas = pd.ExcelFile(
    ARQUIVO_MAPBIOMAS
)

print(
    "Abas encontradas:"
)

for indice, aba in enumerate(
    excel_mapbiomas.sheet_names,
    start=1
):

    print(
        f"{indice}. {aba}"
    )

Abas encontradas:
1. READ_ME
2. COVERAGE_10.1
3. PIVOT_COVERAGE
4. PIVOTCHART_COVERAGE
5. METADADOS
6. LEGEND_CODE


In [7]:
print(
    "Quantidade de abas:"
)

print(
    len(
        excel_mapbiomas.sheet_names
    )
)

Quantidade de abas:
6


In [8]:
amostra_coverage = pd.read_excel(
    ARQUIVO_MAPBIOMAS,
    sheet_name="COVERAGE_10.1",
    nrows=10
)

print(
    "Dimensão da amostra:"
)

print(
    amostra_coverage.shape
)


print(
    "\nColunas:"
)

print(
    amostra_coverage.columns.tolist()
)


display(
    amostra_coverage
)

Dimensão da amostra:
(10, 51)

Colunas:
['country', 'biome', 'state', 'state_acronym', 'municipality', 'class_id', 'class_level_0', 'class_level_1', 'class_level_2', 'class_level_3', 'class_level_4', 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]


,country,biome,state,state_acronym,municipality,class_id,class_level_0,class_level_1,class_level_2,class_level_3,class_level_4,1985,1986,1987,1988,1989,1990,1991,1992,1993,1994,1995,1996,1997,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,Brasil,Amazônia,Acre,AC,Acrelândia,0,Undefined,6. Not Observed,6. Not Observed,6. Not Observed,6. Not Observed,11.712264,11.095658,12.504917,13.825783,13.209385,14.178128,11.447980,12.856795,15.058560,14.001852,13.913573,14.089811,13.913767,15.410259,14.090170,16.380301,16.556377,16.291867,17.172492,16.996621,17.876694,15.763087,14.794616,16.203644,13.297371,15.851566,15.234519,13.825668,16.027521,16.379340,14.178402,16.115742,16.379696,14.794822,14.970546,15.234995,14.002384,14.794761,14.178737,15.234836
1,Brasil,Amazônia,Acre,AC,Acrelândia,3,Natural,1. Forest,1.1. Forest Formation,1.1. Forest Formation,1.1. Forest Formation,169524.604936,169591.113027,166529.385492,164042.851594,162403.854228,160580.215786,159061.898929,156372.457038,153097.477189,149095.118168,146304.749027,138832.398717,135125.818590,130331.776686,123804.167654,119459.160839,114411.933711,105808.965664,98950.336225,93466.548326,87031.347829,81105.184266,77245.297511,74045.222767,70656.065494,67373.621281,64946.484026,63377.706477,62866.011639,60876.136643,59622.392302,58090.821041,57786.236961,57181.401709,55739.605360,53993.083542,52182.020497,49368.132731,46946.282841,45632.716365
2,Brasil,Amazônia,Acre,AC,Acrelândia,6,Natural,1. Forest,1.4 Floodable Forest,1.4 Floodable Forest,1.4 Floodable Forest,1633.961100,1630.790132,1630.702693,1623.394252,1612.035078,1605.696283,1594.247921,1594.160072,1592.486243,1583.239575,1581.831163,1578.484562,1581.125596,1582.971867,1576.895888,1601.993775,1607.189691,1596.976699,1591.429033,1583.150216,1560.524679,1542.123238,1537.457958,1531.909325,1526.450236,1536.313015,1524.866373,1523.895908,1531.376822,1530.054826,1529.614323,1537.012126,1541.061285,1532.344358,1533.047789,1537.978636,1520.461675,1507.597594,1486.986055,1520.181600
3,Brasil,Amazônia,Acre,AC,Acrelândia,11,Natural,2. Non Forest Natural Formation,2.1. Wetland,2.1. Wetland,2.1. Wetland,41.744747,31.267115,23.249456,30.558856,28.357512,32.144127,32.231051,20.343580,18.670420,18.405647,20.519278,23.953843,20.960060,20.870833,20.608587,20.342565,18.405526,18.143327,22.370581,25.101060,27.301549,33.027053,24.924965,23.956131,33.818673,32.673690,32.057500,27.831166,27.567312,23.250005,21.401012,24.130905,24.751571,28.273405,35.753658,46.056572,45.968447,24.835715,19.554040,21.051771
4,Brasil,Amazônia,Acre,AC,Acrelândia,12,Natural,2. Non Forest Natural Formation,2.2. Grassland,2.2. Grassland,2.2. Grassland,190.364096,193.007215,191.069105,175.300515,173.361052,172.833738,167.283957,155.920190,145.612316,134.335701,130.637760,122.006823,113.374174,116.809874,104.830183,108.441632,106.502640,92.757675,88.793258,82.716879,84.213507,78.841773,77.960197,67.565078,56.201103,60.341010,55.232590,48.359763,59.454060,53.464333,54.703320,50.562474,52.763268,40.256034,65.980574,50.297927,86.857863,93.906317,78.226264,36.467106
5,Brasil,Amazônia,Acre,AC,Acrelândia,15,Antropic,3. Farming,3.1. Pasture,3.1. Pasture,3.1. Pasture,8722.915759,8634.381513,11589.494739,13978.304076,15690.143639,17600.039158,19089.818955,21761.643553,25047.730348,29142.857644,31911.197354,39392.362895,43038.519451,47766.529888,54301.285856,58710.472509,63765.866990,72184.797845,79078.834786,84587.557954,91122.993415,97094.956532,101018.244039,104210.675698,107424.271107,110688.746579,113350.383937,114987.082005,115540.977787,117371.776018,118315.799043,119889.642582,120087.817199,121017.195545,122466.834769,124190.179790,125958.178807,128826.347414,131215.864922,132647.444311
6,Brasil,Amazônia,Acre,AC,Acrelândia,24,Antropic,4. Non vegetated area,4.2. Urban Area,4.2. Urban Area,4.2. Urban Area,26.417725,30.556284,33.021822,36.720147,41.827364,46.142036,54.595288,68.860166,79.0

In [9]:
colunas_anos = [
    coluna
    for coluna in amostra_coverage.columns
    if (
        isinstance(
            coluna,
            int
        )
        or
        (
            isinstance(
                coluna,
                str
            )
            and
            coluna.isdigit()
        )
    )
]

print(
    "Primeiros anos encontrados:"
)

print(
    colunas_anos[:10]
)


print(
    "\nÚltimos anos encontrados:"
)

print(
    colunas_anos[-10:]
)


print(
    "\nQuantidade de anos:"
)

print(
    len(
        colunas_anos
    )
)

Primeiros anos encontrados:
[1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994]

Últimos anos encontrados:
[2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

Quantidade de anos:
40


In [10]:
ANOS_PROJETO = [
    2019,
    2020,
    2021,
    2022,
    2023,
    2024
]


for ano in ANOS_PROJETO:

    existe = (
        ano in amostra_coverage.columns
        or
        str(ano)
        in amostra_coverage.columns
    )

    print(
        ano,
        "->",
        existe
    )

2019 -> True
2020 -> True
2021 -> True
2022 -> True
2023 -> True
2024 -> True


## 6.1 — Validação inicial da fonte

A inspeção inicial da aba `COVERAGE_10.1` confirmou a presença de:

- informações territoriais;
- bioma;
- estado e sigla da UF;
- município;
- identificador da classe (`class_id`);
- quatro níveis hierárquicos de classificação;
- séries anuais de área em hectares entre 1985 e 2024.

Todos os anos do escopo do projeto (2019–2024) estão disponíveis.

A partir desta etapa será carregado apenas o conjunto de colunas necessário
para o projeto, preservando o arquivo RAW original sem alterações.

In [12]:
COLUNAS_IDENTIFICACAO = [
    "country",
    "biome",
    "state",
    "state_acronym",
    "municipality",
    "class_id",
    "class_level_0",
    "class_level_1",
    "class_level_2",
    "class_level_3",
    "class_level_4"
]


ANOS_PROJETO = [
    2019,
    2020,
    2021,
    2022,
    2023,
    2024
]


def coluna_interessa_mapbiomas(coluna):

    return (
        coluna in COLUNAS_IDENTIFICACAO
        or
        coluna in ANOS_PROJETO
    )


coverage = pd.read_excel(
    ARQUIVO_MAPBIOMAS,
    sheet_name="COVERAGE_10.1",
    usecols=coluna_interessa_mapbiomas
)


print(
    "Dimensão da base carregada:"
)

print(
    coverage.shape
)


print(
    "\nColunas:"
)

print(
    coverage.columns.tolist()
)


display(
    coverage.head()
)

Dimensão da base carregada:
(78818, 17)

Colunas:
['country', 'biome', 'state', 'state_acronym', 'municipality', 'class_id', 'class_level_0', 'class_level_1', 'class_level_2', 'class_level_3', 'class_level_4', 2019, 2020, 2021, 2022, 2023, 2024]


,country,biome,state,state_acronym,municipality,class_id,class_level_0,class_level_1,class_level_2,class_level_3,class_level_4,2019,2020,2021,2022,2023,2024
0,Brasil,Amazônia,Acre,AC,Acrelândia,0,Undefined,6. Not Observed,6. Not Observed,6. Not Observed,6. Not Observed,14.970546,15.234995,14.002384,14.794761,14.178737,15.234836
1,Brasil,Amazônia,Acre,AC,Acrelândia,3,Natural,1. Forest,1.1. Forest Formation,1.1. Forest Formation,1.1. Forest Formation,55739.605360,53993.083542,52182.020497,49368.132731,46946.282841,45632.716365
2,Brasil,Amazônia,Acre,AC,Acrelândia,6,Natural,1. Forest,1.4 Floodable Forest,1.4 Floodable Forest,1.4 Floodable Forest,1533.047789,1537.978636,1520.461675,1507.597594,1486.986055,1520.181600
3,Brasil,Amazônia,Acre,AC,Acrelândia,11,Natural,2. Non Forest Natural Formation,2.1. Wetland,2.1. Wetland,2.1. Wetland,35.753658,46.056572,45.968447,24.835715,19.554040,21.051771
4,Brasil,Amazônia,Acre,AC,Acrelândia,12,Natural,2. Non Forest Natural Formation,2.2. Grassland,2.2. Grassland,2.2. Grassland,65.980574,50.297927,86.857863,93.906317,78.226264,36.467106


In [13]:
print(
    "Tipos das colunas:"
)

display(
    coverage.dtypes
)


print(
    "\nValores ausentes:"
)

display(
    coverage
    .isna()
    .sum()
)

Tipos das colunas:


country           object
biome             object
state             object
state_acronym     object
municipality      object
class_id           int64
class_level_0     object
class_level_1     object
class_level_2     object
class_level_3     object
class_level_4     object
2019             float64
2020             float64
2021             float64
2022             float64
2023             float64
2024             float64
dtype: object


Valores ausentes:


country          0
biome            0
state            0
state_acronym    0
municipality     0
class_id         0
class_level_0    0
class_level_1    0
class_level_2    0
class_level_3    0
class_level_4    0
2019             0
2020             0
2021             0
2022             0
2023             0
2024             0
dtype: int64

In [14]:
print(
    "Quantidade de UFs:"
)

print(
    coverage[
        "state_acronym"
    ].nunique()
)


print(
    "\nUFs encontradas:"
)

print(
    sorted(
        coverage[
            "state_acronym"
        ]
        .dropna()
        .unique()
    )
)

Quantidade de UFs:
27

UFs encontradas:
['AC', 'AL', 'AM', 'AP', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA', 'MG', 'MS', 'MT', 'PA', 'PB', 'PE', 'PI', 'PR', 'RJ', 'RN', 'RO', 'RR', 'RS', 'SC', 'SE', 'SP', 'TO']


In [15]:
UFS_PROJETO = [
    "DF",
    "GO",
    "MS",
    "MT",
    "PR",
    "RS",
    "SC"
]


coverage_projeto = (
    coverage[
        coverage[
            "state_acronym"
        ].isin(
            UFS_PROJETO
        )
    ]
    .copy()
)


print(
    "Dimensão nacional:"
)

print(
    coverage.shape
)


print(
    "\nDimensão Centro-Oeste + Sul:"
)

print(
    coverage_projeto.shape
)

Dimensão nacional:
(78818, 17)

Dimensão Centro-Oeste + Sul:
(23200, 17)


In [16]:
display(
    coverage_projeto[
        "state_acronym"
    ]
    .value_counts()
    .sort_index()
)

state_acronym
DF      17
GO    3600
MS    1395
MT    2849
PR    5176
RS    6901
SC    3262
Name: count, dtype: int64

In [17]:
municipios_mapbiomas = (
    coverage_projeto[
        [
            "state_acronym",
            "municipality"
        ]
    ]
    .drop_duplicates()
)


print(
    "Total de combinações UF + município:"
)

print(
    len(
        municipios_mapbiomas
    )
)


print(
    "\nMunicípios por UF:"
)

display(
    municipios_mapbiomas[
        "state_acronym"
    ]
    .value_counts()
    .sort_index()
)

Total de combinações UF + município:
1672

Municípios por UF:


state_acronym
DF      1
GO    250
MS     81
MT    144
PR    401
RS    499
SC    296
Name: count, dtype: int64

In [18]:
import geopandas as gpd


MALHA_IBGE_DIR = (
    BASE_DIR
    / "data"
    / "raw"
    / "ibge_territorial"
    / "malha_municipal"
    / "BR_Municipios_2024"
)


arquivos_shp = list(
    MALHA_IBGE_DIR.glob(
        "*.shp"
    )
)


print(
    "Shapefiles encontrados:"
)

print(
    len(arquivos_shp)
)


for arquivo in arquivos_shp:
    print(
        arquivo.name
    )

Shapefiles encontrados:
1
BR_Municipios_2024.shp


In [19]:
MALHA_SHP = arquivos_shp[0]


malha_ibge_2024 = gpd.read_file(
    MALHA_SHP
)


malha_ibge_projeto = (
    malha_ibge_2024[
        malha_ibge_2024[
            "SIGLA_UF"
        ].isin(
            UFS_PROJETO
        )
    ]
    [
        [
            "CD_MUN",
            "NM_MUN",
            "SIGLA_UF"
        ]
    ]
    .copy()
)


print(
    "Municípios da Malha IBGE:"
)

print(
    len(
        malha_ibge_projeto
    )
)

Municípios da Malha IBGE:
1661


In [20]:
import unicodedata
import re


def normalizar_nome_municipio(texto):

    texto = str(
        texto
    ).strip()

    texto = unicodedata.normalize(
        "NFKD",
        texto
    )

    texto = "".join(
        caractere
        for caractere in texto
        if not unicodedata.combining(
            caractere
        )
    )

    texto = texto.lower()

    texto = re.sub(
        r"[^a-z0-9]+",
        " ",
        texto
    )

    texto = re.sub(
        r"\s+",
        " ",
        texto
    )

    return texto.strip()

In [21]:
municipios_mapbiomas_cmp = (
    municipios_mapbiomas
    .copy()
)


municipios_mapbiomas_cmp[
    "municipio_normalizado"
] = (
    municipios_mapbiomas_cmp[
        "municipality"
    ]
    .apply(
        normalizar_nome_municipio
    )
)


malha_ibge_cmp = (
    malha_ibge_projeto
    .copy()
)


malha_ibge_cmp[
    "municipio_normalizado"
] = (
    malha_ibge_cmp[
        "NM_MUN"
    ]
    .apply(
        normalizar_nome_municipio
    )
)

In [22]:
comparacao_municipios = (
    municipios_mapbiomas_cmp
    .merge(
        malha_ibge_cmp,
        left_on=[
            "state_acronym",
            "municipio_normalizado"
        ],
        right_on=[
            "SIGLA_UF",
            "municipio_normalizado"
        ],
        how="outer",
        indicator=True
    )
)

In [23]:
somente_mapbiomas = (
    comparacao_municipios[
        comparacao_municipios[
            "_merge"
        ] == "left_only"
    ]
    .copy()
)


somente_ibge = (
    comparacao_municipios[
        comparacao_municipios[
            "_merge"
        ] == "right_only"
    ]
    .copy()
)


print(
    "Somente no MapBiomas:"
)

print(
    len(
        somente_mapbiomas
    )
)


print(
    "\nSomente na Malha IBGE:"
)

print(
    len(
        somente_ibge
    )
)

Somente no MapBiomas:
14

Somente na Malha IBGE:
3


In [24]:
display(
    somente_mapbiomas[
        [
            "state_acronym",
            "municipality",
            "municipio_normalizado"
        ]
    ]
    .sort_values(
        [
            "state_acronym",
            "municipality"
        ]
    )
)

,state_acronym,municipality,municipio_normalizado
37,GO,Brasília,brasilia
179,GO,Paranã,parana
216,GO,São Desidério,sao desiderio
232,GO,Talismã,talisma
253,MS,Alto Araguaia,alto araguaia
292,MS,Itapura,itapura
346,MT,Aruanã,aruana
407,MT,Nova Crixás,nova crixas
463,MT,Sonora,sonora
596,PR,Florínea,florinea


In [25]:
display(
    somente_ibge[
        [
            "SIGLA_UF",
            "NM_MUN",
            "CD_MUN",
            "municipio_normalizado"
        ]
    ]
    .sort_values(
        [
            "SIGLA_UF",
            "NM_MUN"
        ]
    )
)

,SIGLA_UF,NM_MUN,CD_MUN,municipio_normalizado
350,MT,Boa Esperança do Norte,5101837,boa esperanca do norte
899,RS,"Área Operacional ""Lagoa Mirim""",4300001,area operacional lagoa mirim
898,RS,"Área Operacional ""Lagoa dos Patos""",4300002,area operacional lagoa dos patos


In [26]:
casamentos_corretos = (
    comparacao_municipios[
        comparacao_municipios[
            "_merge"
        ] == "both"
    ]
)


print(
    "Municípios associados diretamente:"
)

print(
    len(
        casamentos_corretos
    )
)


print(
    "\nPercentual do MapBiomas associado:"
)

print(
    round(
        len(
            casamentos_corretos
        )
        /
        len(
            municipios_mapbiomas_cmp
        )
        * 100,
        2
    ),
    "%"
)

Municípios associados diretamente:
1658

Percentual do MapBiomas associado:
99.16 %


In [27]:
nomes_somente_mapbiomas = (
    somente_mapbiomas[
        "municipality"
    ]
    .tolist()
)


malha_ibge_nacional_cmp = (
    malha_ibge_2024[
        [
            "CD_MUN",
            "NM_MUN",
            "SIGLA_UF"
        ]
    ]
    .copy()
)


malha_ibge_nacional_cmp[
    "municipio_normalizado"
] = (
    malha_ibge_nacional_cmp[
        "NM_MUN"
    ]
    .apply(
        normalizar_nome_municipio
    )
)


nomes_normalizados_busca = [
    normalizar_nome_municipio(nome)
    for nome in nomes_somente_mapbiomas
]


candidatos_ibge = (
    malha_ibge_nacional_cmp[
        malha_ibge_nacional_cmp[
            "municipio_normalizado"
        ].isin(
            nomes_normalizados_busca
        )
    ]
    .sort_values(
        [
            "municipio_normalizado",
            "SIGLA_UF"
        ]
    )
)


display(
    candidatos_ibge
)

,CD_MUN,NM_MUN,SIGLA_UF,municipio_normalizado
3993,5100300,Alto Araguaia,MT,alto araguaia
5526,5202502,Aruanã,GO,aruana
2994,5300108,Brasília,DF,brasilia
101,3516101,Florínea,SP,florinea
1128,3523008,Itapura,SP,itapura
1622,3137304,Lagoa dos Patos,MG,lagoa dos patos
3246,4311908,Marcelino Ramos,RS,marcelino ramos
4365,2922102,Mundo Novo,BA,mundo novo
1122,5214051,Mundo Novo,GO,mundo novo
1709,5005681,Mundo Novo,MS,mundo novo


In [28]:
diagnostico_territorial = (
    somente_mapbiomas[
        [
            "state_acronym",
            "municipality",
            "municipio_normalizado"
        ]
    ]
    .merge(
        candidatos_ibge,
        on="municipio_normalizado",
        how="left"
    )
    .sort_values(
        [
            "state_acronym",
            "municipality",
            "SIGLA_UF"
        ]
    )
)


display(
    diagnostico_territorial
)

,state_acronym,municipality,municipio_normalizado,CD_MUN,NM_MUN,SIGLA_UF
0,GO,Brasília,brasilia,5300108,Brasília,DF
1,GO,Paranã,parana,2408607,Paraná,RN
2,GO,Paranã,parana,1716208,Paranã,TO
3,GO,São Desidério,sao desiderio,2928901,São Desidério,BA
4,GO,Talismã,talisma,1720978,Talismã,TO
5,MS,Alto Araguaia,alto araguaia,5100300,Alto Araguaia,MT
6,MS,Itapura,itapura,3523008,Itapura,SP
7,MT,Aruanã,aruana,5202502,Aruanã,GO
8,MT,Nova Crixás,nova crixas,5214838,Nova Crixás,GO
9,MT,Sonora,sonora,5007935,Sonora,MS


In [29]:
nomes_inconsistentes = (
    somente_mapbiomas[
        "municipality"
    ]
    .unique()
)


ocorrencias_nacionais_mapbiomas = (
    coverage[
        coverage[
            "municipality"
        ].isin(
            nomes_inconsistentes
        )
    ][
        [
            "state_acronym",
            "municipality",
            "biome",
            "class_id",
            2019,
            2024
        ]
    ]
    .copy()
)


resumo_ocorrencias = (
    ocorrencias_nacionais_mapbiomas[
        [
            "municipality",
            "state_acronym"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "municipality",
            "state_acronym"
        ]
    )
)


display(
    resumo_ocorrencias
)

,municipality,state_acronym
30438,Alto Araguaia,MS
30474,Alto Araguaia,MT
20060,Aruanã,GO
31742,Aruanã,MT
19691,Brasília,DF
23128,Brasília,GO
52529,Florínea,PR
69656,Florínea,SP
50835,Itapura,MS
70741,Itapura,SP


In [30]:
referencia_ibge_projeto = (
    malha_ibge_projeto[
        [
            "CD_MUN",
            "NM_MUN",
            "SIGLA_UF"
        ]
    ]
    .copy()
)


referencia_ibge_projeto[
    "municipio_normalizado"
] = (
    referencia_ibge_projeto[
        "NM_MUN"
    ]
    .apply(
        normalizar_nome_municipio
    )
)


print(
    "Unidades territoriais IBGE do projeto:"
)

print(
    len(
        referencia_ibge_projeto
    )
)


display(
    referencia_ibge_projeto.head()
)

Unidades territoriais IBGE do projeto:
1661


,CD_MUN,NM_MUN,SIGLA_UF,municipio_normalizado
3,5219902,São Francisco de Goiás,GO,sao francisco de goias
7,4316204,Rondinha,RS,rondinha
8,4317558,Santo Antônio do Palma,RS,santo antonio do palma
9,4209508,Laurentino,SC,laurentino
12,4202107,Barra Velha,SC,barra velha


In [31]:
territorios_mapbiomas_nacional = (
    coverage[
        [
            "state_acronym",
            "municipality"
        ]
    ]
    .drop_duplicates()
    .copy()
)


territorios_mapbiomas_nacional[
    "municipio_normalizado"
] = (
    territorios_mapbiomas_nacional[
        "municipality"
    ]
    .apply(
        normalizar_nome_municipio
    )
)

In [32]:
nomes_ibge_projeto = set(
    referencia_ibge_projeto[
        "municipio_normalizado"
    ]
)


ocorrencias_nomes_projeto = (
    territorios_mapbiomas_nacional[
        territorios_mapbiomas_nacional[
            "municipio_normalizado"
        ].isin(
            nomes_ibge_projeto
        )
    ]
    .copy()
)


print(
    "Ocorrências territoriais encontradas:"
)

print(
    len(
        ocorrencias_nomes_projeto
    )
)


display(
    ocorrencias_nomes_projeto
    .sort_values(
        [
            "municipio_normalizado",
            "state_acronym"
        ]
    )
    .head(50)
)

Ocorrências territoriais encontradas:
1828


,state_acronym,municipality,municipio_normalizado
19713,GO,Abadia de Goiás,abadia de goias
19727,GO,Abadiânia,abadiania
52530,PR,Abatiá,abatia
63160,SC,Abdon Batista,abdon batista
63170,SC,Abelardo Luz,abelardo luz
75926,RS,Aceguá,acegua
30447,MT,Acorizal,acorizal
19744,GO,Acreúna,acreuna
19758,GO,Adelândia,adelandia
52543,PR,Adrianópolis,adrianopolis


In [33]:
diagnostico_fragmentos = (
    ocorrencias_nomes_projeto
    .merge(
        referencia_ibge_projeto,
        on="municipio_normalizado",
        how="inner"
    )
)

In [34]:
fragmentos_uf_diferente = (
    diagnostico_fragmentos[
        diagnostico_fragmentos[
            "state_acronym"
        ]
        !=
        diagnostico_fragmentos[
            "SIGLA_UF"
        ]
    ]
    [
        [
            "state_acronym",
            "municipality",
            "SIGLA_UF",
            "NM_MUN",
            "CD_MUN",
            "municipio_normalizado"
        ]
    ]
    .sort_values(
        [
            "municipality",
            "state_acronym",
            "SIGLA_UF"
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "Combinações com UF diferente:"
)

print(
    len(
        fragmentos_uf_diferente
    )
)


display(
    fragmentos_uf_diferente
)

Combinações com UF diferente:
237


,state_acronym,municipality,SIGLA_UF,NM_MUN,CD_MUN,municipio_normalizado
0,RR,Alto Alegre,RS,Alto Alegre,4300554,alto alegre
1,SP,Alto Alegre,RS,Alto Alegre,4300554,alto alegre
2,MS,Alto Araguaia,MT,Alto Araguaia,5100300,alto araguaia
3,RO,Alto Paraíso,PR,Alto Paraíso,4128625,alto paraiso
4,TO,Alvorada,RS,Alvorada,4300604,alvorada
...,...,...,...,...,...,...
232,SP,Vera Cruz,RS,Vera Cruz,4322707,vera cruz
233,PA,Vila Rica,MT,Vila Rica,5108600,vila rica
234,PI,Várzea Grande,MT,Várzea Grande,5108402,varzea grande
235,MG,Wenceslau Braz,PR,Wenceslau Braz,4128500,wenceslau braz


In [35]:
# ============================================================
# DIAGNÓSTICO ESPACIAL — MUNDO NOVO
# ============================================================

malha_metrica_nacional = (
    malha_ibge_2024
    .to_crs(
        "EPSG:5880"
    )
    .copy()
)


malha_metrica_nacional[
    "municipio_normalizado"
] = (
    malha_metrica_nacional[
        "NM_MUN"
    ]
    .apply(
        normalizar_nome_municipio
    )
)


candidatos_mundo_novo = (
    malha_metrica_nacional[
        malha_metrica_nacional[
            "municipio_normalizado"
        ] == "mundo novo"
    ]
    [
        [
            "CD_MUN",
            "NM_MUN",
            "SIGLA_UF",
            "geometry"
        ]
    ]
    .copy()
)


territorio_pr = (
    malha_metrica_nacional[
        malha_metrica_nacional[
            "SIGLA_UF"
        ] == "PR"
    ]
    .geometry
    .union_all()
)


candidatos_mundo_novo[
    "distancia_ao_parana_km"
] = (
    candidatos_mundo_novo[
        "geometry"
    ]
    .distance(
        territorio_pr
    )
    / 1000
)


display(
    candidatos_mundo_novo[
        [
            "CD_MUN",
            "NM_MUN",
            "SIGLA_UF",
            "distancia_ao_parana_km"
        ]
    ]
    .sort_values(
        "distancia_ao_parana_km"
    )
)

,CD_MUN,NM_MUN,SIGLA_UF,distancia_ao_parana_km
1709,5005681,Mundo Novo,MS,0.000000
1122,5214051,Mundo Novo,GO,963.755633
4365,2922102,Mundo Novo,BA,1544.833655


In [36]:
correcoes_territoriais = pd.DataFrame(
    [
        {
            "state_acronym_mapbiomas": "GO",
            "municipality_mapbiomas": "Brasília",
            "codigo_ibge": "5300108",
            "municipio_ibge": "Brasília",
            "uf_ibge": "DF",
            "acao": "reassociar"
        },
        {
            "state_acronym_mapbiomas": "GO",
            "municipality_mapbiomas": "Paranã",
            "codigo_ibge": pd.NA,
            "municipio_ibge": "Paranã",
            "uf_ibge": "TO",
            "acao": "excluir_fora_escopo"
        },
        {
            "state_acronym_mapbiomas": "GO",
            "municipality_mapbiomas": "São Desidério",
            "codigo_ibge": pd.NA,
            "municipio_ibge": "São Desidério",
            "uf_ibge": "BA",
            "acao": "excluir_fora_escopo"
        },
        {
            "state_acronym_mapbiomas": "GO",
            "municipality_mapbiomas": "Talismã",
            "codigo_ibge": pd.NA,
            "municipio_ibge": "Talismã",
            "uf_ibge": "TO",
            "acao": "excluir_fora_escopo"
        },
        {
            "state_acronym_mapbiomas": "MS",
            "municipality_mapbiomas": "Alto Araguaia",
            "codigo_ibge": "5100300",
            "municipio_ibge": "Alto Araguaia",
            "uf_ibge": "MT",
            "acao": "reassociar"
        },
        {
            "state_acronym_mapbiomas": "MS",
            "municipality_mapbiomas": "Itapura",
            "codigo_ibge": pd.NA,
            "municipio_ibge": "Itapura",
            "uf_ibge": "SP",
            "acao": "excluir_fora_escopo"
        },
        {
            "state_acronym_mapbiomas": "MT",
            "municipality_mapbiomas": "Aruanã",
            "codigo_ibge": "5202502",
            "municipio_ibge": "Aruanã",
            "uf_ibge": "GO",
            "acao": "reassociar"
        },
        {
            "state_acronym_mapbiomas": "MT",
            "municipality_mapbiomas": "Nova Crixás",
            "codigo_ibge": "5214838",
            "municipio_ibge": "Nova Crixás",
            "uf_ibge": "GO",
            "acao": "reassociar"
        },
        {
            "state_acronym_mapbiomas": "MT",
            "municipality_mapbiomas": "Sonora",
            "codigo_ibge": "5007935",
            "municipio_ibge": "Sonora",
            "uf_ibge": "MS",
            "acao": "reassociar"
        },
        {
            "state_acronym_mapbiomas": "PR",
            "municipality_mapbiomas": "Florínea",
            "codigo_ibge": pd.NA,
            "municipio_ibge": "Florínea",
            "uf_ibge": "SP",
            "acao": "excluir_fora_escopo"
        },
        {
            "state_acronym_mapbiomas": "PR",
            "municipality_mapbiomas": "Mundo Novo",
            "codigo_ibge": "5005681",
            "municipio_ibge": "Mundo Novo",
            "uf_ibge": "MS",
            "acao": "reassociar"
        },
        {
            "state_acronym_mapbiomas": "RS",
            "municipality_mapbiomas": "Lagoa Mirim",
            "codigo_ibge": "4300001",
            "municipio_ibge": 'Área Operacional "Lagoa Mirim"',
            "uf_ibge": "RS",
            "acao": "reassociar_area_operacional"
        },
        {
            "state_acronym_mapbiomas": "RS",
            "municipality_mapbiomas": "Lagoa dos Patos",
            "codigo_ibge": "4300002",
            "municipio_ibge": 'Área Operacional "Lagoa dos Patos"',
            "uf_ibge": "RS",
            "acao": "reassociar_area_operacional"
        },
        {
            "state_acronym_mapbiomas": "SC",
            "municipality_mapbiomas": "Marcelino Ramos",
            "codigo_ibge": "4311908",
            "municipio_ibge": "Marcelino Ramos",
            "uf_ibge": "RS",
            "acao": "reassociar"
        }
    ]
)


print(
    "Quantidade de exceções:"
)

print(
    len(correcoes_territoriais)
)


print(
    "\nAções:"
)

display(
    correcoes_territoriais[
        "acao"
    ]
    .value_counts()
)


display(
    correcoes_territoriais
)

Quantidade de exceções:
14

Ações:


acao
reassociar                     7
excluir_fora_escopo            5
reassociar_area_operacional    2
Name: count, dtype: int64

,state_acronym_mapbiomas,municipality_mapbiomas,codigo_ibge,municipio_ibge,uf_ibge,acao
0,GO,Brasília,5300108,Brasília,DF,reassociar
1,GO,Paranã,<NA>,Paranã,TO,excluir_fora_escopo
2,GO,São Desidério,<NA>,São Desidério,BA,excluir_fora_escopo
3,GO,Talismã,<NA>,Talismã,TO,excluir_fora_escopo
4,MS,Alto Araguaia,5100300,Alto Araguaia,MT,reassociar
5,MS,Itapura,<NA>,Itapura,SP,excluir_fora_escopo
6,MT,Aruanã,5202502,Aruanã,GO,reassociar
7,MT,Nova Crixás,5214838,Nova Crixás,GO,reassociar
8,MT,Sonora,5007935,Sonora,MS,reassociar
9,PR,Florínea,<NA>,Florínea,SP,excluir_fora_escopo


In [37]:
ARQUIVO_CORRECOES_TERRITORIAIS = (
    QUALITY_MAPBIOMAS_COBERTURA_DIR
    / "correcoes_territoriais_mapbiomas_ibge_2024.csv"
)


correcoes_territoriais.to_csv(
    ARQUIVO_CORRECOES_TERRITORIAIS,
    index=False,
    encoding="utf-8-sig"
)


print(
    "Arquivo salvo:"
)

print(
    ARQUIVO_CORRECOES_TERRITORIAIS
)

Arquivo salvo:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\databases_processed\mapbiomas_cobertura\quality\correcoes_territoriais_mapbiomas_ibge_2024.csv


In [38]:
# ============================================================
# APLICAÇÃO DA REFERÊNCIA TERRITORIAL IBGE
# ============================================================

coverage_territorial = (
    coverage_projeto
    .copy()
)


coverage_territorial[
    "municipio_normalizado"
] = (
    coverage_territorial[
        "municipality"
    ]
    .apply(
        normalizar_nome_municipio
    )
)


referencia_merge = (
    referencia_ibge_projeto[
        [
            "CD_MUN",
            "NM_MUN",
            "SIGLA_UF",
            "municipio_normalizado"
        ]
    ]
    .copy()
)


referencia_merge[
    "CD_MUN"
] = (
    referencia_merge[
        "CD_MUN"
    ]
    .astype("string")
    .str.zfill(7)
)


coverage_territorial = (
    coverage_territorial
    .merge(
        referencia_merge,
        left_on=[
            "state_acronym",
            "municipio_normalizado"
        ],
        right_on=[
            "SIGLA_UF",
            "municipio_normalizado"
        ],
        how="left"
    )
)


print(
    "Dimensão após merge:"
)

print(
    coverage_territorial.shape
)


print(
    "\nLinhas com correspondência direta:"
)

print(
    coverage_territorial[
        "CD_MUN"
    ]
    .notna()
    .sum()
)


print(
    "\nLinhas sem correspondência direta:"
)

print(
    coverage_territorial[
        "CD_MUN"
    ]
    .isna()
    .sum()
)

Dimensão após merge:
(23200, 21)

Linhas com correspondência direta:
23149

Linhas sem correspondência direta:
51


In [39]:
correcoes_merge = (
    correcoes_territoriais
    .copy()
)


correcoes_merge[
    "codigo_ibge"
] = (
    correcoes_merge[
        "codigo_ibge"
    ]
    .astype("string")
)


coverage_territorial = (
    coverage_territorial
    .merge(
        correcoes_merge,
        left_on=[
            "state_acronym",
            "municipality"
        ],
        right_on=[
            "state_acronym_mapbiomas",
            "municipality_mapbiomas"
        ],
        how="left"
    )
)

In [40]:
coverage_territorial[
    "codigo_ibge_final"
] = (
    coverage_territorial[
        "CD_MUN"
    ]
    .combine_first(
        coverage_territorial[
            "codigo_ibge"
        ]
    )
)


coverage_territorial[
    "municipio_ibge_final"
] = (
    coverage_territorial[
        "NM_MUN"
    ]
    .combine_first(
        coverage_territorial[
            "municipio_ibge"
        ]
    )
)


coverage_territorial[
    "uf_ibge_final"
] = (
    coverage_territorial[
        "SIGLA_UF"
    ]
    .combine_first(
        coverage_territorial[
            "uf_ibge"
        ]
    )
)


coverage_territorial[
    "acao_territorial"
] = (
    coverage_territorial[
        "acao"
    ]
    .fillna(
        "correspondencia_direta"
    )
)

In [41]:
print(
    "Ações territoriais:"
)

display(
    coverage_territorial[
        "acao_territorial"
    ]
    .value_counts()
)


nao_resolvidos = (
    coverage_territorial[
        coverage_territorial[
            "codigo_ibge_final"
        ].isna()
        &
        (
            coverage_territorial[
                "acao_territorial"
            ]
            !=
            "excluir_fora_escopo"
        )
    ]
)


print(
    "\nLinhas sem código IBGE e que NÃO são exclusões:"
)

print(
    len(
        nao_resolvidos
    )
)

Ações territoriais:


acao_territorial
correspondencia_direta         23149
reassociar_area_operacional       26
reassociar                        18
excluir_fora_escopo                7
Name: count, dtype: int64


Linhas sem código IBGE e que NÃO são exclusões:
0


In [42]:
unidades_resolvidas = (
    coverage_territorial[
        coverage_territorial[
            "acao_territorial"
        ]
        !=
        "excluir_fora_escopo"
    ][
        [
            "codigo_ibge_final",
            "municipio_ibge_final",
            "uf_ibge_final"
        ]
    ]
    .drop_duplicates()
)


print(
    "Unidades territoriais representadas:"
)

print(
    len(
        unidades_resolvidas
    )
)


display(
    unidades_resolvidas[
        "uf_ibge_final"
    ]
    .value_counts()
    .sort_index()
)

Unidades territoriais representadas:
1660


uf_ibge_final
DF      1
GO    246
MS     79
MT    141
PR    399
RS    499
SC    295
Name: count, dtype: int64

In [43]:
codigos_mapbiomas = set(
    unidades_resolvidas[
        "codigo_ibge_final"
    ]
    .dropna()
    .astype(str)
)


referencia_codigos = (
    referencia_ibge_projeto
    .copy()
)


referencia_codigos[
    "CD_MUN"
] = (
    referencia_codigos[
        "CD_MUN"
    ]
    .astype("string")
    .str.zfill(7)
)


unidades_ausentes_mapbiomas = (
    referencia_codigos[
        ~referencia_codigos[
            "CD_MUN"
        ].isin(
            codigos_mapbiomas
        )
    ]
)


print(
    "Quantidade de unidades da Malha sem representação MapBiomas:"
)

print(
    len(
        unidades_ausentes_mapbiomas
    )
)


display(
    unidades_ausentes_mapbiomas[
        [
            "CD_MUN",
            "NM_MUN",
            "SIGLA_UF"
        ]
    ]
)

Quantidade de unidades da Malha sem representação MapBiomas:
1


,CD_MUN,NM_MUN,SIGLA_UF
2575,5101837,Boa Esperança do Norte,MT


In [44]:
coverage_territorial_valido = (
    coverage_territorial[
        coverage_territorial[
            "acao_territorial"
        ] != "excluir_fora_escopo"
    ]
    .copy()
)


print(
    "Linhas antes da exclusão:"
)

print(
    len(
        coverage_territorial
    )
)


print(
    "\nLinhas válidas para o projeto:"
)

print(
    len(
        coverage_territorial_valido
    )
)


print(
    "\nUnidades territoriais:"
)

print(
    coverage_territorial_valido[
        "codigo_ibge_final"
    ].nunique()
)

Linhas antes da exclusão:
23200

Linhas válidas para o projeto:
23193

Unidades territoriais:
1660


In [45]:
print(
    "Biomas encontrados:"
)

print(
    sorted(
        coverage_territorial_valido[
            "biome"
        ]
        .dropna()
        .unique()
    )
)


print(
    "\nQuantidade de linhas por bioma:"
)

display(
    coverage_territorial_valido[
        "biome"
    ]
    .value_counts()
)

Biomas encontrados:
['Amazônia', 'Cerrado', 'Mata Atlântica', 'Pampa', 'Pantanal']

Quantidade de linhas por bioma:


biome
Mata Atlântica    13158
Cerrado            5713
Pampa              2682
Amazônia           1404
Pantanal            236
Name: count, dtype: int64

In [46]:
biomas_por_unidade = (
    coverage_territorial_valido[
        [
            "codigo_ibge_final",
            "municipio_ibge_final",
            "uf_ibge_final",
            "biome"
        ]
    ]
    .drop_duplicates()
    .groupby(
        [
            "codigo_ibge_final",
            "municipio_ibge_final",
            "uf_ibge_final"
        ],
        as_index=False
    )
    .agg(
        quantidade_biomas=(
            "biome",
            "nunique"
        )
    )
)


print(
    "Distribuição da quantidade de biomas por unidade:"
)

display(
    biomas_por_unidade[
        "quantidade_biomas"
    ]
    .value_counts()
    .sort_index()
)


print(
    "\nUnidades presentes em mais de um bioma:"
)

print(
    (
        biomas_por_unidade[
            "quantidade_biomas"
        ] > 1
    ).sum()
)

Distribuição da quantidade de biomas por unidade:


quantidade_biomas
1    1395
2     264
3       1
Name: count, dtype: int64


Unidades presentes em mais de um bioma:
265


In [47]:
unidades_multibioma = (
    biomas_por_unidade[
        biomas_por_unidade[
            "quantidade_biomas"
        ] > 1
    ]
    .copy()
)


display(
    unidades_multibioma
    .sort_values(
        [
            "quantidade_biomas",
            "uf_ibge_final",
            "municipio_ibge_final"
        ],
        ascending=[
            False,
            True,
            True
        ]
    )
    .head(50)
)

,codigo_ibge_final,municipio_ibge_final,uf_ibge_final,quantidade_biomas
1291,5102504,Cáceres,MT,3
1451,5203906,Buriti Alegre,GO,2
1455,5204102,Cachoeira Alta,GO,2
1457,5204250,Cachoeira Dourada,GO,2
1460,5204508,Caldas Novas,GO,2
1458,5204300,Caçu,GO,2
1482,5205901,Corumbaíba,GO,2
1487,5206602,Cumari,GO,2
1511,5209150,Gouvelândia,GO,2
1520,5209937,Inaciolândia,GO,2


In [48]:
detalhe_multibioma = (
    coverage_territorial_valido[
        coverage_territorial_valido[
            "codigo_ibge_final"
        ].isin(
            unidades_multibioma[
                "codigo_ibge_final"
            ]
        )
    ][
        [
            "codigo_ibge_final",
            "municipio_ibge_final",
            "uf_ibge_final",
            "biome"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "uf_ibge_final",
            "municipio_ibge_final",
            "biome"
        ]
    )
)


display(
    detalhe_multibioma.head(100)
)

,codigo_ibge_final,municipio_ibge_final,uf_ibge_final,biome
1936,5203906,Buriti Alegre,GO,Cerrado
7133,5203906,Buriti Alegre,GO,Mata Atlântica
1985,5204102,Cachoeira Alta,GO,Cerrado
7144,5204102,Cachoeira Alta,GO,Mata Atlântica
2015,5204250,Cachoeira Dourada,GO,Cerrado
7152,5204250,Cachoeira Dourada,GO,Mata Atlântica
2056,5204508,Caldas Novas,GO,Cerrado
7172,5204508,Caldas Novas,GO,Mata Atlântica
2027,5204300,Caçu,GO,Cerrado
7163,5204300,Caçu,GO,Mata Atlântica


In [49]:
# ============================================================
# RESUMO DE BIOMAS POR UNIDADE TERRITORIAL
# ============================================================

resumo_biomas_unidade = (
    coverage_territorial_valido[
        [
            "codigo_ibge_final",
            "municipio_ibge_final",
            "uf_ibge_final",
            "biome"
        ]
    ]
    .drop_duplicates()
    .groupby(
        [
            "codigo_ibge_final",
            "municipio_ibge_final",
            "uf_ibge_final"
        ],
        as_index=False
    )
    .agg(
        quantidade_biomas=(
            "biome",
            "nunique"
        ),
        biomas_presentes=(
            "biome",
            lambda serie: " | ".join(
                sorted(
                    serie.dropna().unique()
                )
            )
        )
    )
)


print(
    "Dimensão:"
)

print(
    resumo_biomas_unidade.shape
)


display(
    resumo_biomas_unidade[
        resumo_biomas_unidade[
            "quantidade_biomas"
        ] > 1
    ]
    .head(20)
)

Dimensão:
(1660, 5)


,codigo_ibge_final,municipio_ibge_final,uf_ibge_final,quantidade_biomas,biomas_presentes
18,4101606,Arapoti,PR,2,Cerrado | Mata Atlântica
69,4104907,Castro,PR,2,Cerrado | Mata Atlântica
168,4112009,Jaguariaíva,PR,2,Cerrado | Mata Atlântica
268,4119400,Piraí do Sul,PR,2,Cerrado | Mata Atlântica
347,4125407,São José da Boa Vista,PR,2,Cerrado | Mata Atlântica
361,4126306,Sengés,PR,2,Cerrado | Mata Atlântica
372,4127106,Telêmaco Borba,PR,2,Cerrado | Mata Atlântica
376,4127502,Tibagi,PR,2,Cerrado | Mata Atlântica
390,4128500,Wenceslau Braz,PR,2,Cerrado | Mata Atlântica
391,4128534,Ventania,PR,2,Cerrado | Mata Atlântica


In [50]:
display(
    resumo_biomas_unidade[
        resumo_biomas_unidade[
            "quantidade_biomas"
        ] == 3
    ]
)

,codigo_ibge_final,municipio_ibge_final,uf_ibge_final,quantidade_biomas,biomas_presentes
1291,5102504,Cáceres,MT,3,Amazônia | Cerrado | Pantanal


In [51]:
ARQUIVO_AUDITORIA_BIOMAS = (
    QUALITY_MAPBIOMAS_COBERTURA_DIR
    / "auditoria_biomas_por_unidade_2019_2024.csv"
)


resumo_biomas_unidade.to_csv(
    ARQUIVO_AUDITORIA_BIOMAS,
    index=False,
    encoding="utf-8-sig"
)


print(
    ARQUIVO_AUDITORIA_BIOMAS
)

C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\databases_processed\mapbiomas_cobertura\quality\auditoria_biomas_por_unidade_2019_2024.csv


In [52]:
legend_code = pd.read_excel(
    ARQUIVO_MAPBIOMAS,
    sheet_name="LEGEND_CODE"
)


print(
    "Dimensão LEGEND_CODE:"
)

print(
    legend_code.shape
)


print(
    "\nColunas:"
)

print(
    legend_code.columns.tolist()
)


display(
    legend_code.head(50)
)

Dimensão LEGEND_CODE:
(41, 6)

Colunas:
['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5']


,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5
0,NaN,NaN,Códigos das classes da legenda da Coleção 10 d...,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,COLEÇÃO 10 - CLASSES,COLLECTION 10 - CLASSES,Code ID,Hexacode Number,Color ID
3,NaN,1. Floresta,1. Forest,1,#1f8d49,NaN
4,NaN,1.1 Formação Florestal,1.1. Forest Formation,3,#1f8d49,NaN
5,NaN,1.2. Formação Savânica,1.2. Savanna Formation,4,#7dc975,NaN
6,NaN,1.3. Mangue,1.3. Mangrove,5,#04381d,NaN
7,NaN,1.4. Floresta Alagável,1.4 Floodable Forest,6,#026975,NaN
8,NaN,1.5. Restinga Arbórea,1.5. Wooded Sandbank Vegetation,49,#02d659,NaN
9,NaN,2. Vegetação Herbácea e Arbustiva,2. Herbaceous and Shrubby Vegetation,10,#ad975a,NaN


In [53]:
classes_projeto = (
    coverage_territorial_valido[
        [
            "class_id",
            "class_level_0",
            "class_level_1",
            "class_level_2",
            "class_level_3",
            "class_level_4"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "class_id",
            "class_level_0",
            "class_level_1",
            "class_level_2",
            "class_level_3",
            "class_level_4"
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "Quantidade de class_id:"
)

print(
    classes_projeto[
        "class_id"
    ].nunique()
)


print(
    "\nQuantidade de combinações de classificação:"
)

print(
    len(
        classes_projeto
    )
)


display(
    classes_projeto
)

Quantidade de class_id:
30

Quantidade de combinações de classificação:
30


,class_id,class_level_0,class_level_1,class_level_2,class_level_3,class_level_4
0,0,Undefined,6. Not Observed,6. Not Observed,6. Not Observed,6. Not Observed
1,3,Natural,1. Forest,1.1. Forest Formation,1.1. Forest Formation,1.1. Forest Formation
2,4,Natural,1. Forest,1.2. Savanna Formation,1.2. Savanna Formation,1.2. Savanna Formation
3,5,Natural,1. Forest,1.3. Mangrove,1.3. Mangrove,1.3. Mangrove
4,6,Natural,1. Forest,1.4 Floodable Forest,1.4 Floodable Forest,1.4 Floodable Forest
5,9,Antropic,3. Farming,3.3. Forest Plantation,3.3. Forest Plantation,3.3. Forest Plantation
6,11,Natural,2. Non Forest Natural Formation,2.1. Wetland,2.1. Wetland,2.1. Wetland
7,12,Natural,2. Non Forest Natural Formation,2.2. Grassland,2.2. Grassland,2.2. Grassland
8,13,Natural,2. Non Forest Natural Formation,2.6. Other non Forest Formations,2.6. Other non Forest Formations,2.6. Other non Forest Formations
9,15,Antropic,3. Farming,3.1. Pasture,3.1. Pasture,3.1. Pasture


In [54]:
classes_projeto = (
    coverage_territorial_valido[
        [
            "class_id",
            "class_level_0",
            "class_level_1",
            "class_level_2",
            "class_level_3",
            "class_level_4"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "class_id",
            "class_level_0",
            "class_level_1",
            "class_level_2",
            "class_level_3",
            "class_level_4"
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "Quantidade de class_id:"
)

print(
    classes_projeto[
        "class_id"
    ].nunique()
)


print(
    "\nQuantidade de combinações de classificação:"
)

print(
    len(
        classes_projeto
    )
)


display(
    classes_projeto
)

Quantidade de class_id:
30

Quantidade de combinações de classificação:
30


,class_id,class_level_0,class_level_1,class_level_2,class_level_3,class_level_4
0,0,Undefined,6. Not Observed,6. Not Observed,6. Not Observed,6. Not Observed
1,3,Natural,1. Forest,1.1. Forest Formation,1.1. Forest Formation,1.1. Forest Formation
2,4,Natural,1. Forest,1.2. Savanna Formation,1.2. Savanna Formation,1.2. Savanna Formation
3,5,Natural,1. Forest,1.3. Mangrove,1.3. Mangrove,1.3. Mangrove
4,6,Natural,1. Forest,1.4 Floodable Forest,1.4 Floodable Forest,1.4 Floodable Forest
5,9,Antropic,3. Farming,3.3. Forest Plantation,3.3. Forest Plantation,3.3. Forest Plantation
6,11,Natural,2. Non Forest Natural Formation,2.1. Wetland,2.1. Wetland,2.1. Wetland
7,12,Natural,2. Non Forest Natural Formation,2.2. Grassland,2.2. Grassland,2.2. Grassland
8,13,Natural,2. Non Forest Natural Formation,2.6. Other non Forest Formations,2.6. Other non Forest Formations,2.6. Other non Forest Formations
9,15,Antropic,3. Farming,3.1. Pasture,3.1. Pasture,3.1. Pasture


In [55]:
# ============================================================
# METADADOS DA BASE MAPBIOMAS
# ============================================================

metadados_mapbiomas = pd.read_excel(
    ARQUIVO_MAPBIOMAS,
    sheet_name="METADADOS",
    header=2
)


metadados_mapbiomas = (
    metadados_mapbiomas
    .drop(
        columns=[
            coluna
            for coluna in metadados_mapbiomas.columns
            if str(coluna).startswith("Unnamed")
        ],
        errors="ignore"
    )
)


print(
    "Dimensão METADADOS:"
)

print(
    metadados_mapbiomas.shape
)


print(
    "\nColunas:"
)

print(
    metadados_mapbiomas.columns.tolist()
)


display(
    metadados_mapbiomas
)

Dimensão METADADOS:
(10, 3)

Colunas:
['Campo/Field', 'Descrição do campo', 'Field Description']


,Campo/Field,Descrição do campo,Field Description
0,country,País,Country
1,biome,Bioma,Biome
2,state,Estado,State
3,state_acronym,Sigla do estado,State abbreviation
4,class_id,Código da legenda correspondente a classe de c...,Legend code for each land cover and land use c...
5,class_level_0,Classe de cobertura e uso da terra no nível 0,Land cover and land use class on legend level 0
6,class_level_1,Classe de cobertura e uso da terra no nível 1,Land cover and land use class on legend level 1
7,class_level_2,Classe de cobertura e uso da terra no nível 2,Land cover and land use class on legend level 2
8,class_level_3,Classe de cobertura e uso da terra no nível 3,Land cover and land use class on legend level 3
9,class_level_4,Classe de cobertura e uso da terra no nível 4,Land cover and land use class on legend level 4


In [56]:
# ============================================================
# LEGENDA OFICIAL — VERSÃO LIMPA
# ============================================================

legend_code_clean = pd.read_excel(
    ARQUIVO_MAPBIOMAS,
    sheet_name="LEGEND_CODE",
    header=3
)


legend_code_clean = (
    legend_code_clean
    .drop(
        columns=[
            coluna
            for coluna in legend_code_clean.columns
            if str(coluna).startswith("Unnamed")
        ],
        errors="ignore"
    )
    .rename(
        columns={
            "COLEÇÃO 10 - CLASSES": "classe_pt",
            "COLLECTION 10 - CLASSES": "classe_en",
            "Code ID": "class_id",
            "Hexacode Number": "hexadecimal",
            "Color ID": "color_id"
        }
    )
)


legend_code_clean[
    "class_id"
] = pd.to_numeric(
    legend_code_clean[
        "class_id"
    ],
    errors="coerce"
).astype("Int64")


print(
    "Dimensão da legenda limpa:"
)

print(
    legend_code_clean.shape
)


print(
    "\nColunas:"
)

print(
    legend_code_clean.columns.tolist()
)


display(
    legend_code_clean
)

Dimensão da legenda limpa:
(38, 5)

Colunas:
['classe_pt', 'classe_en', 'class_id', 'hexadecimal', 'color_id']


,classe_pt,classe_en,class_id,hexadecimal,color_id
0,1. Floresta,1. Forest,1,#1f8d49,NaN
1,1.1 Formação Florestal,1.1. Forest Formation,3,#1f8d49,NaN
2,1.2. Formação Savânica,1.2. Savanna Formation,4,#7dc975,NaN
3,1.3. Mangue,1.3. Mangrove,5,#04381d,NaN
4,1.4. Floresta Alagável,1.4 Floodable Forest,6,#026975,NaN
5,1.5. Restinga Arbórea,1.5. Wooded Sandbank Vegetation,49,#02d659,NaN
6,2. Vegetação Herbácea e Arbustiva,2. Herbaceous and Shrubby Vegetation,10,#ad975a,NaN
7,2.1. Campo Alagado e Área Pantanosa,2.1. Wetland,11,#519799,NaN
8,2.2. Formação Campestre,2.2. Grassland,12,#d6bc74,NaN
9,2.3. Apicum,2.3. Hypersaline Tidal Flat,32,#fc8114,NaN


In [57]:
codigos_coverage_projeto = set(
    classes_projeto[
        "class_id"
    ]
    .astype(int)
)


codigos_legenda = set(
    legend_code_clean[
        "class_id"
    ]
    .dropna()
    .astype(int)
)


somente_coverage = sorted(
    codigos_coverage_projeto
    -
    codigos_legenda
)


somente_legenda = sorted(
    codigos_legenda
    -
    codigos_coverage_projeto
)


print(
    "Códigos presentes no COVERAGE e ausentes na LEGEND_CODE:"
)

print(
    somente_coverage
)


print(
    "\nCódigos presentes na LEGEND_CODE e ausentes no nosso recorte:"
)

print(
    somente_legenda
)

Códigos presentes no COVERAGE e ausentes na LEGEND_CODE:
[0, 13]

Códigos presentes na LEGEND_CODE e ausentes no nosso recorte:
[1, 10, 14, 18, 19, 22, 26, 27, 35, 36]


In [58]:
# ============================================================
# AUDITORIA DAS CLASSES MAPBIOMAS
# ============================================================

ARQUIVO_CLASSES_PROJETO = (
    QUALITY_MAPBIOMAS_COBERTURA_DIR
    / "classes_mapbiomas_presentes_centro_oeste_sul.csv"
)


classes_projeto.to_csv(
    ARQUIVO_CLASSES_PROJETO,
    index=False,
    encoding="utf-8-sig"
)


ARQUIVO_LEGENDA_LIMPA = (
    QUALITY_MAPBIOMAS_COBERTURA_DIR
    / "legenda_mapbiomas_colecao_10_1_limpa.csv"
)


legend_code_clean.to_csv(
    ARQUIVO_LEGENDA_LIMPA,
    index=False,
    encoding="utf-8-sig"
)


print(
    "Classes do projeto:"
)

print(
    ARQUIVO_CLASSES_PROJETO
)


print(
    "\nLegenda limpa:"
)

print(
    ARQUIVO_LEGENDA_LIMPA
)

Classes do projeto:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\databases_processed\mapbiomas_cobertura\quality\classes_mapbiomas_presentes_centro_oeste_sul.csv

Legenda limpa:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\databases_processed\mapbiomas_cobertura\quality\legenda_mapbiomas_colecao_10_1_limpa.csv


In [59]:
# ============================================================
# CLASSIFICAÇÃO ANALÍTICA DAS CLASSES
# ============================================================

classes_analiticas = (
    classes_projeto
    .copy()
)


classes_analiticas[
    "eh_floresta"
] = (
    classes_analiticas[
        "class_level_1"
    ] == "1. Forest"
)


classes_analiticas[
    "eh_formacao_natural_nao_florestal"
] = (
    classes_analiticas[
        "class_level_1"
    ] == "2. Non Forest Natural Formation"
)


classes_analiticas[
    "eh_agropecuaria"
] = (
    classes_analiticas[
        "class_level_1"
    ] == "3. Farming"
)


classes_analiticas[
    "eh_pastagem"
] = (
    classes_analiticas[
        "class_level_2"
    ] == "3.1. Pasture"
)


classes_analiticas[
    "eh_agricultura"
] = (
    classes_analiticas[
        "class_level_2"
    ] == "3.2. Agriculture"
)


classes_analiticas[
    "eh_lavoura_temporaria"
] = (
    classes_analiticas[
        "class_level_3"
    ] == "3.2.1. Temporary Crop"
)


classes_analiticas[
    "eh_soja"
] = (
    classes_analiticas[
        "class_level_4"
    ] == "3.2.1.1. Soybean"
)


classes_analiticas[
    "eh_silvicultura"
] = (
    classes_analiticas[
        "class_level_2"
    ] == "3.3. Forest Plantation"
)


classes_analiticas[
    "eh_mosaico_usos"
] = (
    classes_analiticas[
        "class_level_2"
    ] == "3.4. Mosaic of Uses"
)


classes_analiticas[
    "eh_area_nao_vegetada"
] = (
    classes_analiticas[
        "class_level_1"
    ] == "4. Non vegetated area"
)


classes_analiticas[
    "eh_corpo_agua"
] = (
    classes_analiticas[
        "class_level_1"
    ] == "5. Water and Marine Environment"
)


classes_analiticas[
    "eh_nao_observado"
] = (
    classes_analiticas[
        "class_id"
    ] == 0
)


display(
    classes_analiticas
)

,class_id,class_level_0,class_level_1,class_level_2,class_level_3,class_level_4,eh_floresta,eh_formacao_natural_nao_florestal,eh_agropecuaria,eh_pastagem,eh_agricultura,eh_lavoura_temporaria,eh_soja,eh_silvicultura,eh_mosaico_usos,eh_area_nao_vegetada,eh_corpo_agua,eh_nao_observado
0,0,Undefined,6. Not Observed,6. Not Observed,6. Not Observed,6. Not Observed,False,False,False,False,False,False,False,False,False,False,False,True
1,3,Natural,1. Forest,1.1. Forest Formation,1.1. Forest Formation,1.1. Forest Formation,True,False,False,False,False,False,False,False,False,False,False,False
2,4,Natural,1. Forest,1.2. Savanna Formation,1.2. Savanna Formation,1.2. Savanna Formation,True,False,False,False,False,False,False,False,False,False,False,False
3,5,Natural,1. Forest,1.3. Mangrove,1.3. Mangrove,1.3. Mangrove,True,False,False,False,False,False,False,False,False,False,False,False
4,6,Natural,1. Forest,1.4 Floodable Forest,1.4 Floodable Forest,1.4 Floodable Forest,True,False,False,False,False,False,False,False,False,False,False,False
5,9,Antropic,3. Farming,3.3. Forest Plantation,3.3. Forest Plantation,3.3. Forest Plantation,False,False,True,False,False,False,False,True,False,False,False,False
6,11,Natural,2. Non Forest Natural Formation,2.1. Wetland,2.1. Wetland,2.1. Wetland,False,True,False,False,False,False,False,False,False,False,False,False
7,12,Natural,2. Non Forest Natural Formation,2.2. Grassland,2.2. Grassland,2.2. Grassland,False,True,False,False,False,False,False,False,False,False,False,False
8,13,Natural,2. Non Forest Natural Formation,2.6. Other non Forest Formations,2.6. Other non Forest Formations,2.6. Other non Forest Formations,False,True,False,False,False,False,False,False,False,False,False,False
9,15,Antropic,3. Farming,3.1. Pasture,3.1. Pasture,3.1. Pasture,False,False,True,True,False,False,False,False,False,False,False,False


In [60]:
flags_analiticas = [
    "eh_floresta",
    "eh_formacao_natural_nao_florestal",
    "eh_agropecuaria",
    "eh_pastagem",
    "eh_agricultura",
    "eh_lavoura_temporaria",
    "eh_soja",
    "eh_silvicultura",
    "eh_mosaico_usos",
    "eh_area_nao_vegetada",
    "eh_corpo_agua",
    "eh_nao_observado"
]


resumo_flags = []


for flag in flags_analiticas:

    ids = (
        classes_analiticas.loc[
            classes_analiticas[
                flag
            ],
            "class_id"
        ]
        .tolist()
    )

    resumo_flags.append(
        {
            "indicador": flag,
            "quantidade_classes": len(ids),
            "class_ids": ids
        }
    )


resumo_flags = pd.DataFrame(
    resumo_flags
)


display(
    resumo_flags
)

,indicador,quantidade_classes,class_ids
0,eh_floresta,5,"[3, 4, 5, 6, 49]"
1,eh_formacao_natural_nao_florestal,6,"[11, 12, 13, 29, 32, 50]"
2,eh_agropecuaria,11,"[9, 15, 20, 21, 39, 40, 41, 46, 47, 48, 62]"
3,eh_pastagem,1,[15]
4,eh_agricultura,8,"[20, 39, 40, 41, 46, 47, 48, 62]"
5,eh_lavoura_temporaria,5,"[20, 39, 40, 41, 62]"
6,eh_soja,1,[39]
7,eh_silvicultura,1,[9]
8,eh_mosaico_usos,1,[21]
9,eh_area_nao_vegetada,5,"[23, 24, 25, 30, 75]"


In [61]:
# ============================================================
# AJUSTE DOS INDICADORES AQUÁTICOS
# ============================================================

classes_analiticas[
    "eh_agua_natural"
] = (
    classes_analiticas[
        "class_id"
    ] == 33
)


classes_analiticas[
    "eh_aquicultura"
] = (
    classes_analiticas[
        "class_id"
    ] == 31
)


classes_analiticas[
    "eh_ambiente_aquatico"
] = (
    classes_analiticas[
        "class_level_1"
    ] == "5. Water and Marine Environment"
)


display(
    classes_analiticas[
        [
            "class_id",
            "class_level_0",
            "class_level_1",
            "class_level_2",
            "eh_agua_natural",
            "eh_aquicultura",
            "eh_ambiente_aquatico"
        ]
    ][
        classes_analiticas[
            "eh_ambiente_aquatico"
        ]
    ]
)

,class_id,class_level_0,class_level_1,class_level_2,eh_agua_natural,eh_aquicultura,eh_ambiente_aquatico
17,31,Antropic,5. Water and Marine Environment,5.2. Aquaculture,False,True,True
19,33,Natural,5. Water and Marine Environment,"5.1. River, Lake and Ocean",True,False,True


## 6.2 — Estrutura analítica das classes

Após a auditoria territorial e da legenda do MapBiomas, foram identificadas
30 classes operacionais de cobertura e uso da terra no recorte do projeto.

As classes foram organizadas em indicadores hierárquicos, incluindo:

- floresta;
- formação natural não florestal;
- agropecuária;
- pastagem;
- agricultura;
- lavoura temporária;
- soja;
- silvicultura;
- mosaico de usos;
- área não vegetada;
- água natural;
- aquicultura;
- área não observada.

As categorias hierárquicas não devem ser somadas entre si, pois classes
mais específicas estão contidas em categorias superiores.

A partir desta etapa, os dados anuais de 2019 a 2024 serão transformados
do formato largo para o formato longo.

In [62]:
# ============================================================
# ASSOCIAR INDICADORES ANALÍTICOS À BASE DE COBERTURA
# ============================================================

FLAGS_FINAIS = [
    "eh_floresta",
    "eh_formacao_natural_nao_florestal",
    "eh_agropecuaria",
    "eh_pastagem",
    "eh_agricultura",
    "eh_lavoura_temporaria",
    "eh_soja",
    "eh_silvicultura",
    "eh_mosaico_usos",
    "eh_area_nao_vegetada",
    "eh_agua_natural",
    "eh_aquicultura",
    "eh_nao_observado"
]


mapa_classes_analiticas = (
    classes_analiticas[
        [
            "class_id",
            *FLAGS_FINAIS
        ]
    ]
    .copy()
)


coverage_classificado = (
    coverage_territorial_valido
    .merge(
        mapa_classes_analiticas,
        on="class_id",
        how="left",
        validate="many_to_one"
    )
)


print(
    "Dimensão antes:"
)

print(
    coverage_territorial_valido.shape
)


print(
    "\nDimensão após classificação:"
)

print(
    coverage_classificado.shape
)

Dimensão antes:
(23193, 31)

Dimensão após classificação:
(23193, 44)


In [63]:
print(
    "Linhas com alguma flag ausente:"
)

print(
    coverage_classificado[
        FLAGS_FINAIS
    ]
    .isna()
    .any(axis=1)
    .sum()
)

Linhas com alguma flag ausente:
0


In [64]:
# ============================================================
# TRANSFORMAÇÃO WIDE -> LONG
# ============================================================

COLUNAS_ID_LONG = [
    "codigo_ibge_final",
    "municipio_ibge_final",
    "uf_ibge_final",
    "biome",
    "class_id",
    "class_level_0",
    "class_level_1",
    "class_level_2",
    "class_level_3",
    "class_level_4",
    "acao_territorial",
    *FLAGS_FINAIS
]


coverage_long = (
    coverage_classificado
    .melt(
        id_vars=COLUNAS_ID_LONG,
        value_vars=ANOS_PROJETO,
        var_name="ano",
        value_name="area_ha"
    )
)


coverage_long[
    "ano"
] = pd.to_numeric(
    coverage_long[
        "ano"
    ],
    errors="raise"
).astype(int)


coverage_long[
    "area_ha"
] = pd.to_numeric(
    coverage_long[
        "area_ha"
    ],
    errors="coerce"
)


print(
    "Dimensão da base longa:"
)

print(
    coverage_long.shape
)


display(
    coverage_long.head(10)
)

Dimensão da base longa:
(139158, 26)


,codigo_ibge_final,municipio_ibge_final,uf_ibge_final,biome,class_id,class_level_0,class_level_1,class_level_2,class_level_3,class_level_4,acao_territorial,eh_floresta,eh_formacao_natural_nao_florestal,eh_agropecuaria,eh_pastagem,eh_agricultura,eh_lavoura_temporaria,eh_soja,eh_silvicultura,eh_mosaico_usos,eh_area_nao_vegetada,eh_agua_natural,eh_aquicultura,eh_nao_observado,ano,area_ha
0,5100201,Água Boa,MT,Amazônia,3,Natural,1. Forest,1.1. Forest Formation,1.1. Forest Formation,1.1. Forest Formation,correspondencia_direta,True,False,False,False,False,False,False,False,False,False,False,False,False,2019,110.931745
1,5100201,Água Boa,MT,Amazônia,4,Natural,1. Forest,1.2. Savanna Formation,1.2. Savanna Formation,1.2. Savanna Formation,correspondencia_direta,True,False,False,False,False,False,False,False,False,False,False,False,False,2019,48.859344
2,5100201,Água Boa,MT,Amazônia,11,Natural,2. Non Forest Natural Formation,2.1. Wetland,2.1. Wetland,2.1. Wetland,correspondencia_direta,False,True,False,False,False,False,False,False,False,False,False,False,False,2019,0.608580
3,5100201,Água Boa,MT,Amazônia,12,Natural,2. Non Forest Natural Formation,2.2. Grassland,2.2. Grassland,2.2. Grassland,correspondencia_direta,False,True,False,False,False,False,False,False,False,False,False,False,False,2019,10.258719
4,5100201,Água Boa,MT,Amazônia,15,Antropic,3. Farming,3.1. Pasture,3.1. Pasture,3.1. Pasture,correspondencia_direta,False,False,True,True,False,False,False,False,False,False,False,False,False,2019,2.260353
5,5100201,Água Boa,MT,Amazônia,33,Natural,5. Water and Marine Environment,"5.1. River, Lake and Ocean","5.1. River, Lake and Ocean","5.1. River, Lake and Ocean",correspondencia_direta,False,False,False,False,False,False,False,False,False,False,True,False,False,2019,6.607400
6,5100250,Alta Floresta,MT,Amazônia,0,Undefined,6. Not Observed,6. Not Observed,6. Not Observed,6. Not Observed,correspondencia_direta,False,False,False,False,False,False,False,False,False,False,False,False,True,2019,0.000000
7,5100250,Alta Floresta,MT,Amazônia,3,Natural,1. Forest,1.1. Forest Formation,1.1. Forest Formation,1.1. Forest Formation,correspondencia_direta,True,False,False,False,False,False,False,False,False,False,False,False,False,2019,390246.067876
8,5100250,Alta Floresta,MT,Amazônia,4,Natural,1. Forest,1.2. Savanna Formation,1.2. Savanna Formation,1.2. Savanna Formation,correspondencia_direta,True,False,False,False,False,False,False,False,False,False,False,False,False,2019,335.777567
9,5100250,Alta Floresta,MT,Amazônia,6,Natural,1. Forest,1.4 Floodable Forest,1.4 Floodable Forest,1.4 Floodable Forest,correspondencia_direta,True,False,False,False,False,False,False,False,False,False,False,False,False,2019,39957.625368


In [65]:
# ============================================================
# VALIDAÇÃO DA BASE LONGA
# ============================================================

print(
    "Período:"
)

print(
    sorted(
        coverage_long[
            "ano"
        ].unique()
    )
)


print(
    "\nQuantidade de anos:"
)

print(
    coverage_long[
        "ano"
    ].nunique()
)


print(
    "\nÁreas ausentes:"
)

print(
    coverage_long[
        "area_ha"
    ].isna().sum()
)


print(
    "\nÁreas negativas:"
)

print(
    (
        coverage_long[
            "area_ha"
        ] < 0
    ).sum()
)


print(
    "\nUnidades territoriais:"
)

print(
    coverage_long[
        "codigo_ibge_final"
    ].nunique()
)


print(
    "\nClass IDs:"
)

print(
    coverage_long[
        "class_id"
    ].nunique()
)

Período:
[np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

Quantidade de anos:
6

Áreas ausentes:
0

Áreas negativas:
0

Unidades territoriais:
1660

Class IDs:
30


In [66]:
# ============================================================
# DIAGNÓSTICO — MUNICÍPIO × ANO × CLASSE
# ============================================================

CHAVE_MUNICIPIO_CLASSE_ANO = [
    "codigo_ibge_final",
    "ano",
    "class_id"
]


duplicidades_classe = (
    coverage_long
    .groupby(
        CHAVE_MUNICIPIO_CLASSE_ANO,
        as_index=False
    )
    .size()
    .rename(
        columns={
            "size": "quantidade_registros"
        }
    )
)


print(
    "Combinações município × ano × classe:"
)

print(
    len(
        duplicidades_classe
    )
)


print(
    "\nCombinações representadas por mais de uma linha:"
)

print(
    (
        duplicidades_classe[
            "quantidade_registros"
        ] > 1
    ).sum()
)


print(
    "\nMáximo de registros para a mesma combinação:"
)

print(
    duplicidades_classe[
        "quantidade_registros"
    ].max()
)


print(
    "\nDistribuição:"
)

display(
    duplicidades_classe[
        "quantidade_registros"
    ]
    .value_counts()
    .sort_index()
)

Combinações município × ano × classe:
123096

Combinações representadas por mais de uma linha:
15966

Máximo de registros para a mesma combinação:
3

Distribuição:


quantidade_registros
1    107130
2     15870
3        96
Name: count, dtype: int64

In [67]:
exemplos_multiplas_parcelas = (
    duplicidades_classe[
        duplicidades_classe[
            "quantidade_registros"
        ] > 1
    ]
    .sort_values(
        "quantidade_registros",
        ascending=False
    )
    .head(20)
)


display(
    exemplos_multiplas_parcelas
)

,codigo_ibge_final,ano,class_id,quantidade_registros
89323,5007935,2023,41,3
91465,5102504,2021,11,3
91484,5102504,2022,11,3
91483,5102504,2022,9,3
91482,5102504,2022,6,3
91481,5102504,2022,4,3
91480,5102504,2022,3,3
91477,5102504,2021,41,3
91476,5102504,2021,39,3
91475,5102504,2021,33,3


In [68]:
# ============================================================
# AGREGAÇÃO MUNICIPAL DAS CLASSES
# ============================================================

COLUNAS_AGRUPAMENTO_CLASSE = [
    "codigo_ibge_final",
    "municipio_ibge_final",
    "uf_ibge_final",
    "ano",
    "class_id",
    "class_level_0",
    "class_level_1",
    "class_level_2",
    "class_level_3",
    "class_level_4",
    *FLAGS_FINAIS
]


coverage_municipio_classe_ano = (
    coverage_long
    .groupby(
        COLUNAS_AGRUPAMENTO_CLASSE,
        as_index=False,
        dropna=False
    )
    .agg(
        area_ha=(
            "area_ha",
            "sum"
        ),
        quantidade_registros_origem=(
            "biome",
            "size"
        ),
        quantidade_biomas_origem=(
            "biome",
            "nunique"
        ),
        biomas_origem=(
            "biome",
            lambda serie: " | ".join(
                sorted(
                    serie.dropna().unique()
                )
            )
        )
    )
)


print(
    "Dimensão após agregação municipal:"
)

print(
    coverage_municipio_classe_ano.shape
)


display(
    coverage_municipio_classe_ano.head(10)
)

Dimensão após agregação municipal:
(123096, 27)


,codigo_ibge_final,municipio_ibge_final,uf_ibge_final,ano,class_id,class_level_0,class_level_1,class_level_2,class_level_3,class_level_4,eh_floresta,eh_formacao_natural_nao_florestal,eh_agropecuaria,eh_pastagem,eh_agricultura,eh_lavoura_temporaria,eh_soja,eh_silvicultura,eh_mosaico_usos,eh_area_nao_vegetada,eh_agua_natural,eh_aquicultura,eh_nao_observado,area_ha,quantidade_registros_origem,quantidade_biomas_origem,biomas_origem
0,4100103,Abatiá,PR,2019,3,Natural,1. Forest,1.1. Forest Formation,1.1. Forest Formation,1.1. Forest Formation,True,False,False,False,False,False,False,False,False,False,False,False,False,2110.560484,1,1,Mata Atlântica
1,4100103,Abatiá,PR,2019,9,Antropic,3. Farming,3.3. Forest Plantation,3.3. Forest Plantation,3.3. Forest Plantation,False,False,True,False,False,False,False,True,False,False,False,False,False,155.260906,1,1,Mata Atlântica
2,4100103,Abatiá,PR,2019,11,Natural,2. Non Forest Natural Formation,2.1. Wetland,2.1. Wetland,2.1. Wetland,False,True,False,False,False,False,False,False,False,False,False,False,False,20.153876,1,1,Mata Atlântica
3,4100103,Abatiá,PR,2019,15,Antropic,3. Farming,3.1. Pasture,3.1. Pasture,3.1. Pasture,False,False,True,True,False,False,False,False,False,False,False,False,False,5773.710590,1,1,Mata Atlântica
4,4100103,Abatiá,PR,2019,20,Antropic,3. Farming,3.2. Agriculture,3.2.1. Temporary Crop,3.2.1.2. Sugar cane,False,False,True,False,True,True,False,False,False,False,False,False,False,426.420699,1,1,Mata Atlântica
5,4100103,Abatiá,PR,2019,21,Antropic,3. Farming,3.4. Mosaic of Uses,3.4. Mosaic of Uses,3.4. Mosaic of Uses,False,False,True,False,False,False,False,False,True,False,False,False,False,6077.606191,1,1,Mata Atlântica
6,4100103,Abatiá,PR,2019,24,Antropic,4. Non vegetated area,4.2. Urban Area,4.2. Urban Area,4.2. Urban Area,False,False,False,False,False,False,False,False,False,True,False,False,False,149.084403,1,1,Mata Atlântica
7,4100103,Abatiá,PR,2019,25,Natural/Antropic,4. Non vegetated area,4.5. Other non Vegetated Areas,4.5. Other non Vegetated Areas,4.5. Other non Vegetated Areas,False,False,False,False,False,False,False,False,False,True,False,False,False,10.447852,1,1,Mata Atlântica
8,4100103,Abatiá,PR,2019,33,Natural,5. Water and Marine Environment,"5.1. River, Lake and Ocean","5.1. River, Lake and Ocean","5.1. River, Lake and Ocean",False,False,False,False,False,False,False,False,False,False,True,False,False,80.725196,1,1,Mata Atlântica
9,4100103,Abatiá,PR,2019,39,Antropic,3. Farming,3.2. Agriculture,3.2.1. Temporary Crop,3.2.1.1. Soybean,False,False,True,False,True,True,True,False,False,False,False,False,False,6337.048921,1,1,Mata Atlântica


In [69]:
# ============================================================
# VALIDAÇÃO DA AGREGAÇÃO MUNICIPAL
# ============================================================

duplicatas_pos_agregacao = (
    coverage_municipio_classe_ano
    .duplicated(
        subset=[
            "codigo_ibge_final",
            "ano",
            "class_id"
        ]
    )
    .sum()
)


print(
    "Duplicatas código IBGE + ano + classe:"
)

print(
    duplicatas_pos_agregacao
)


print(
    "\nUnidades territoriais:"
)

print(
    coverage_municipio_classe_ano[
        "codigo_ibge_final"
    ].nunique()
)


print(
    "\nAnos:"
)

print(
    sorted(
        coverage_municipio_classe_ano[
            "ano"
        ].unique()
    )
)


print(
    "\nClasses:"
)

print(
    coverage_municipio_classe_ano[
        "class_id"
    ].nunique()
)


print(
    "\nÁreas ausentes:"
)

print(
    coverage_municipio_classe_ano[
        "area_ha"
    ].isna().sum()
)


print(
    "\nÁreas negativas:"
)

print(
    (
        coverage_municipio_classe_ano[
            "area_ha"
        ] < 0
    ).sum()
)

Duplicatas código IBGE + ano + classe:
0

Unidades territoriais:
1660

Anos:
[np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

Classes:
30

Áreas ausentes:
0

Áreas negativas:
0


In [70]:
area_antes = (
    coverage_long[
        "area_ha"
    ]
    .sum()
)


area_depois = (
    coverage_municipio_classe_ano[
        "area_ha"
    ]
    .sum()
)


diferenca_area = (
    area_depois
    -
    area_antes
)


print(
    "\nÁrea total antes da agregação:"
)

print(
    area_antes
)


print(
    "\nÁrea total depois da agregação:"
)

print(
    area_depois
)


print(
    "\nDiferença:"
)

print(
    diferenca_area
)


Área total antes da agregação:
1309474586.9263077

Área total depois da agregação:
1309474586.9263077

Diferença:
0.0


In [71]:
# ============================================================
# SALVAR PROCESSED — MUNICÍPIO × ANO × CLASSE
# ============================================================

ARQUIVO_PROCESSED_CLASSE = (
    PROCESSED_MAPBIOMAS_COBERTURA_DIR
    / "mapbiomas_cobertura_municipio_classe_ano_2019_2024.csv"
)


coverage_municipio_classe_ano.to_csv(
    ARQUIVO_PROCESSED_CLASSE,
    index=False,
    encoding="utf-8-sig"
)


print(
    "Arquivo Processed salvo:"
)

print(
    ARQUIVO_PROCESSED_CLASSE
)


print(
    "\nDimensão:"
)

print(
    coverage_municipio_classe_ano.shape
)

Arquivo Processed salvo:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\databases_processed\mapbiomas_cobertura\mapbiomas_cobertura_municipio_classe_ano_2019_2024.csv

Dimensão:
(123096, 27)


In [72]:
# ============================================================
# INDICADORES MUNICIPAIS DE COBERTURA E USO DA TERRA
# ============================================================

coverage_indicadores_base = (
    coverage_municipio_classe_ano
    .copy()
)


MAPA_INDICADORES = {
    "eh_floresta": "area_floresta_ha",
    "eh_formacao_natural_nao_florestal": "area_formacao_natural_nao_florestal_ha",
    "eh_agropecuaria": "area_agropecuaria_ha",
    "eh_pastagem": "area_pastagem_ha",
    "eh_agricultura": "area_agricultura_ha",
    "eh_lavoura_temporaria": "area_lavoura_temporaria_ha",
    "eh_soja": "area_soja_mapbiomas_ha",
    "eh_silvicultura": "area_silvicultura_ha",
    "eh_mosaico_usos": "area_mosaico_usos_ha",
    "eh_area_nao_vegetada": "area_nao_vegetada_ha",
    "eh_agua_natural": "area_agua_natural_ha",
    "eh_aquicultura": "area_aquicultura_ha",
    "eh_nao_observado": "area_nao_observada_ha"
}


for flag, coluna_area in MAPA_INDICADORES.items():

    coverage_indicadores_base[
        coluna_area
    ] = (
        coverage_indicadores_base[
            "area_ha"
        ]
        *
        coverage_indicadores_base[
            flag
        ].astype(int)
    )

In [73]:
COLUNAS_AREAS_INDICADORES = list(
    MAPA_INDICADORES.values()
)


mapbiomas_municipio_ano = (
    coverage_indicadores_base
    .groupby(
        [
            "codigo_ibge_final",
            "municipio_ibge_final",
            "uf_ibge_final",
            "ano"
        ],
        as_index=False
    )
    .agg(
        {
            **{
                coluna: "sum"
                for coluna in COLUNAS_AREAS_INDICADORES
            },
            "area_ha": "sum"
        }
    )
    .rename(
        columns={
            "area_ha": "area_total_mapeada_ha"
        }
    )
)


print(
    "Dimensão município × ano:"
)

print(
    mapbiomas_municipio_ano.shape
)


display(
    mapbiomas_municipio_ano.head(10)
)

Dimensão município × ano:
(9960, 18)


,codigo_ibge_final,municipio_ibge_final,uf_ibge_final,ano,area_floresta_ha,area_formacao_natural_nao_florestal_ha,area_agropecuaria_ha,area_pastagem_ha,area_agricultura_ha,area_lavoura_temporaria_ha,area_soja_mapbiomas_ha,area_silvicultura_ha,area_mosaico_usos_ha,area_nao_vegetada_ha,area_agua_natural_ha,area_aquicultura_ha,area_nao_observada_ha,area_total_mapeada_ha
0,4100103,Abatiá,PR,2019,2110.560484,20.153876,20501.078533,5773.710590,8494.500847,8183.880715,6337.048921,155.260906,6077.606191,159.532255,80.725196,0.0,0.0,22872.050344
1,4100103,Abatiá,PR,2020,2138.038478,24.102157,20465.703400,5515.560753,8557.179526,8245.983106,6814.108692,153.615340,6239.347781,163.892775,80.313534,0.0,0.0,22872.050344
2,4100103,Abatiá,PR,2021,2144.043562,24.842192,20457.066441,5136.433064,8573.384341,8262.928519,6855.572934,153.039484,6594.209551,165.620277,80.477873,0.0,0.0,22872.050344
3,4100103,Abatiá,PR,2022,2120.430937,27.228733,20463.566371,4904.415611,8601.355462,8291.722262,6969.448836,152.628175,6805.167123,176.562297,84.262006,0.0,0.0,22872.050344
4,4100103,Abatiá,PR,2023,2111.213707,26.652856,20467.189193,4644.000071,8617.563791,8307.766021,7127.089229,152.134601,7053.490730,180.100145,86.894442,0.0,0.0,22872.050344
5,4100103,Abatiá,PR,2024,2117.137267,26.817535,20433.207767,4371.896706,8662.977838,8353.015562,7320.511082,152.710527,7245.622696,209.803791,85.083983,0.0,0.0,22872.050344
6,4100202,Adrianópolis,PR,2019,85989.071184,50.016002,48265.330411,10506.257320,80.791269,80.791269,50.119234,9474.065756,28204.216067,181.495746,446.973528,0.0,0.0,134932.886871
7,4100202,Adrianópolis,PR,2020,85989.690424,51.563552,48273.413323,10602.989266,80.384820,80.384820,27.505013,9527.322185,28062.717051,202.017929,416.201642,0.0,0.0,134932.886871
8,4100202,Adrianópolis,PR,2021,85930.105205,47.985500,48324.768238,10688.603686,81.768153,81.768153,29.784209,9556.042463,27998.353935,217.327138,412.700790,0.0,0.0,134932.886871
9,4100202,Adrianópolis,PR,2022,86090.728184,43.999357,48114.151000,10697.337270,83.394620,83.394620,29.784212,9592.973806,27740.445304,244.034910,439.973420,0.0,0.0,134932.886871


In [74]:
# ============================================================
# COBERTURA NATURAL + INFORMAÇÃO DE BIOMAS
# ============================================================

mapbiomas_municipio_ano[
    "area_cobertura_natural_ha"
] = (
    mapbiomas_municipio_ano[
        "area_floresta_ha"
    ]
    +
    mapbiomas_municipio_ano[
        "area_formacao_natural_nao_florestal_ha"
    ]
)

In [75]:
mapbiomas_municipio_ano = (
    mapbiomas_municipio_ano
    .merge(
        resumo_biomas_unidade,
        left_on=[
            "codigo_ibge_final",
            "municipio_ibge_final",
            "uf_ibge_final"
        ],
        right_on=[
            "codigo_ibge_final",
            "municipio_ibge_final",
            "uf_ibge_final"
        ],
        how="left",
        validate="many_to_one"
    )
)


print(
    "Dimensão após inclusão dos biomas:"
)

print(
    mapbiomas_municipio_ano.shape
)


display(
    mapbiomas_municipio_ano.head()
)

Dimensão após inclusão dos biomas:
(9960, 21)


,codigo_ibge_final,municipio_ibge_final,uf_ibge_final,ano,area_floresta_ha,area_formacao_natural_nao_florestal_ha,area_agropecuaria_ha,area_pastagem_ha,area_agricultura_ha,area_lavoura_temporaria_ha,area_soja_mapbiomas_ha,area_silvicultura_ha,area_mosaico_usos_ha,area_nao_vegetada_ha,area_agua_natural_ha,area_aquicultura_ha,area_nao_observada_ha,area_total_mapeada_ha,area_cobertura_natural_ha,quantidade_biomas,biomas_presentes
0,4100103,Abatiá,PR,2019,2110.560484,20.153876,20501.078533,5773.710590,8494.500847,8183.880715,6337.048921,155.260906,6077.606191,159.532255,80.725196,0.0,0.0,22872.050344,2130.714359,1,Mata Atlântica
1,4100103,Abatiá,PR,2020,2138.038478,24.102157,20465.703400,5515.560753,8557.179526,8245.983106,6814.108692,153.615340,6239.347781,163.892775,80.313534,0.0,0.0,22872.050344,2162.140635,1,Mata Atlântica
2,4100103,Abatiá,PR,2021,2144.043562,24.842192,20457.066441,5136.433064,8573.384341,8262.928519,6855.572934,153.039484,6594.209551,165.620277,80.477873,0.0,0.0,22872.050344,2168.885754,1,Mata Atlântica
3,4100103,Abatiá,PR,2022,2120.430937,27.228733,20463.566371,4904.415611,8601.355462,8291.722262,6969.448836,152.628175,6805.167123,176.562297,84.262006,0.0,0.0,22872.050344,2147.659670,1,Mata Atlântica
4,4100103,Abatiá,PR,2023,2111.213707,26.652856,20467.189193,4644.000071,8617.563791,8307.766021,7127.089229,152.134601,7053.490730,180.100145,86.894442,0.0,0.0,22872.050344,2137.866563,1,Mata Atlântica


In [76]:
# ============================================================
# VALIDAÇÃO MUNICÍPIO × ANO
# ============================================================

print(
    "Dimensão:"
)

print(
    mapbiomas_municipio_ano.shape
)


print(
    "\nUnidades territoriais:"
)

print(
    mapbiomas_municipio_ano[
        "codigo_ibge_final"
    ].nunique()
)


print(
    "\nPeríodo:"
)

print(
    sorted(
        mapbiomas_municipio_ano[
            "ano"
        ].unique()
    )
)


print(
    "\nDuplicatas código IBGE + ano:"
)

print(
    mapbiomas_municipio_ano
    .duplicated(
        subset=[
            "codigo_ibge_final",
            "ano"
        ]
    )
    .sum()
)

Dimensão:
(9960, 21)

Unidades territoriais:
1660

Período:
[np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

Duplicatas código IBGE + ano:
0


In [77]:
print(
    "\nQuantidade de registros por ano:"
)

display(
    mapbiomas_municipio_ano[
        "ano"
    ]
    .value_counts()
    .sort_index()
)


Quantidade de registros por ano:


ano
2019    1660
2020    1660
2021    1660
2022    1660
2023    1660
2024    1660
Name: count, dtype: int64

In [78]:
print(
    "\nValores ausentes nos indicadores:"
)

display(
    mapbiomas_municipio_ano[
        COLUNAS_AREAS_INDICADORES
        +
        [
            "area_cobertura_natural_ha",
            "area_total_mapeada_ha"
        ]
    ]
    .isna()
    .sum()
)


print(
    "\nValores negativos:"
)

for coluna in (
    COLUNAS_AREAS_INDICADORES
    +
    [
        "area_cobertura_natural_ha",
        "area_total_mapeada_ha"
    ]
):

    quantidade_negativos = (
        mapbiomas_municipio_ano[
            coluna
        ] < 0
    ).sum()

    print(
        coluna,
        "->",
        quantidade_negativos
    )


Valores ausentes nos indicadores:


area_floresta_ha                          0
area_formacao_natural_nao_florestal_ha    0
area_agropecuaria_ha                      0
area_pastagem_ha                          0
area_agricultura_ha                       0
area_lavoura_temporaria_ha                0
area_soja_mapbiomas_ha                    0
area_silvicultura_ha                      0
area_mosaico_usos_ha                      0
area_nao_vegetada_ha                      0
area_agua_natural_ha                      0
area_aquicultura_ha                       0
area_nao_observada_ha                     0
area_cobertura_natural_ha                 0
area_total_mapeada_ha                     0
dtype: int64


Valores negativos:
area_floresta_ha -> 0
area_formacao_natural_nao_florestal_ha -> 0
area_agropecuaria_ha -> 0
area_pastagem_ha -> 0
area_agricultura_ha -> 0
area_lavoura_temporaria_ha -> 0
area_soja_mapbiomas_ha -> 0
area_silvicultura_ha -> 0
area_mosaico_usos_ha -> 0
area_nao_vegetada_ha -> 0
area_agua_natural_ha -> 0
area_aquicultura_ha -> 0
area_nao_observada_ha -> 0
area_cobertura_natural_ha -> 0
area_total_mapeada_ha -> 0


In [79]:
# ============================================================
# AUDITORIA — ESTABILIDADE DA ÁREA TOTAL MAPEADA
# ============================================================

auditoria_area_temporal = (
    mapbiomas_municipio_ano
    .groupby(
        [
            "codigo_ibge_final",
            "municipio_ibge_final",
            "uf_ibge_final"
        ],
        as_index=False
    )
    .agg(
        area_total_min_ha=(
            "area_total_mapeada_ha",
            "min"
        ),
        area_total_max_ha=(
            "area_total_mapeada_ha",
            "max"
        ),
        area_total_media_ha=(
            "area_total_mapeada_ha",
            "mean"
        )
    )
)


auditoria_area_temporal[
    "amplitude_area_ha"
] = (
    auditoria_area_temporal[
        "area_total_max_ha"
    ]
    -
    auditoria_area_temporal[
        "area_total_min_ha"
    ]
)


auditoria_area_temporal[
    "amplitude_area_pct"
] = (
    auditoria_area_temporal[
        "amplitude_area_ha"
    ]
    /
    auditoria_area_temporal[
        "area_total_media_ha"
    ]
    * 100
)


print(
    "Unidades analisadas:"
)

print(
    len(
        auditoria_area_temporal
    )
)


print(
    "\nResumo da amplitude em hectares:"
)

display(
    auditoria_area_temporal[
        "amplitude_area_ha"
    ]
    .describe()
)


print(
    "\nMaior variação percentual:"
)

print(
    auditoria_area_temporal[
        "amplitude_area_pct"
    ].max()
)

Unidades analisadas:
1660

Resumo da amplitude em hectares:


count    1.660000e+03
mean     1.974236e-08
std      1.218718e-07
min      1.091394e-11
25%      2.469278e-10
50%      9.385985e-10
75%      3.492460e-09
max      1.703196e-06
Name: amplitude_area_ha, dtype: float64


Maior variação percentual:
7.276272681502697e-09


In [80]:
display(
    auditoria_area_temporal
    .sort_values(
        "amplitude_area_pct",
        ascending=False
    )
    .head(20)
)

,codigo_ibge_final,municipio_ibge_final,uf_ibge_final,area_total_min_ha,area_total_max_ha,area_total_media_ha,amplitude_area_ha,amplitude_area_pct
22,4101853,Ariranha do Ivaí,PR,23407.536393,23407.536395,23407.536395,1.703196e-06,7.276273e-09
296,4121356,Rancho Alegre D'Oeste,PR,24140.050558,24140.050559,24140.050559,1.038919e-06,4.303717e-09
112,4107801,Floraí,PR,19108.415819,19108.415819,19108.415819,6.349364e-07,3.322810e-09
338,4124608,São Carlos do Ivaí,PR,22525.136592,22525.136593,22525.136593,6.775772e-07,3.008094e-09
236,4117206,Nova Olímpia,PR,13632.013182,13632.013183,13632.013183,3.357382e-07,2.462866e-09
76,4105607,Cidade Gaúcha,PR,40303.485391,40303.485392,40303.485392,8.851202e-07,2.196138e-09
179,4112959,Juranda,PR,35429.730046,35429.730046,35429.730046,7.086928e-07,2.000277e-09
1588,5215900,Palminópolis,GO,39331.632363,39331.632364,39331.632364,6.366899e-07,1.618773e-09
162,4111506,Ivaiporã,PR,43697.763036,43697.763037,43697.763037,5.567781e-07,1.274157e-09
1217,5003157,Coronel Sapucaia,MS,102374.539491,102374.539492,102374.539492,1.102642e-06,1.077067e-09


In [81]:
# ============================================================
# REFERÊNCIA DE ÁREA — MALHA MUNICIPAL IBGE 2024
# ============================================================

area_ibge_referencia = (
    malha_ibge_2024[
        malha_ibge_2024[
            "SIGLA_UF"
        ].isin(
            UFS_PROJETO
        )
    ][
        [
            "CD_MUN",
            "NM_MUN",
            "SIGLA_UF",
            "AREA_KM2"
        ]
    ]
    .copy()
)


area_ibge_referencia[
    "CD_MUN"
] = (
    area_ibge_referencia[
        "CD_MUN"
    ]
    .astype("string")
    .str.zfill(7)
)


area_ibge_referencia[
    "area_ibge_ha"
] = (
    pd.to_numeric(
        area_ibge_referencia[
            "AREA_KM2"
        ],
        errors="coerce"
    )
    * 100
)


print(
    "Unidades na referência IBGE:"
)

print(
    len(
        area_ibge_referencia
    )
)


print(
    "\nÁrea IBGE ausente:"
)

print(
    area_ibge_referencia[
        "area_ibge_ha"
    ]
    .isna()
    .sum()
)


display(
    area_ibge_referencia.head()
)

Unidades na referência IBGE:
1661

Área IBGE ausente:
0


,CD_MUN,NM_MUN,SIGLA_UF,AREA_KM2,area_ibge_ha
3,5219902,São Francisco de Goiás,GO,416.535,41653.5
7,4316204,Rondinha,RS,252.454,25245.4
8,4317558,Santo Antônio do Palma,RS,126.094,12609.4
9,4209508,Laurentino,SC,79.333,7933.3
12,4202107,Barra Velha,SC,138.947,13894.7


In [82]:
# ============================================================
# COMPARAÇÃO DE ÁREA — MAPBIOMAS × IBGE
# ============================================================

comparacao_area_ibge = (
    auditoria_area_temporal
    .merge(
        area_ibge_referencia[
            [
                "CD_MUN",
                "AREA_KM2",
                "area_ibge_ha"
            ]
        ],
        left_on="codigo_ibge_final",
        right_on="CD_MUN",
        how="left",
        validate="one_to_one"
    )
)


comparacao_area_ibge[
    "diferenca_area_ha"
] = (
    comparacao_area_ibge[
        "area_total_media_ha"
    ]
    -
    comparacao_area_ibge[
        "area_ibge_ha"
    ]
)


comparacao_area_ibge[
    "diferenca_area_abs_ha"
] = (
    comparacao_area_ibge[
        "diferenca_area_ha"
    ]
    .abs()
)


comparacao_area_ibge[
    "diferenca_area_pct"
] = (
    comparacao_area_ibge[
        "diferenca_area_ha"
    ]
    /
    comparacao_area_ibge[
        "area_ibge_ha"
    ]
    * 100
)


comparacao_area_ibge[
    "diferenca_area_abs_pct"
] = (
    comparacao_area_ibge[
        "diferenca_area_pct"
    ]
    .abs()
)


print(
    "Unidades comparadas:"
)

print(
    len(
        comparacao_area_ibge
    )
)


print(
    "\nÁreas IBGE ausentes após merge:"
)

print(
    comparacao_area_ibge[
        "area_ibge_ha"
    ]
    .isna()
    .sum()
)


print(
    "\nResumo da diferença absoluta percentual:"
)

display(
    comparacao_area_ibge[
        "diferenca_area_abs_pct"
    ]
    .describe()
)

Unidades comparadas:
1660

Áreas IBGE ausentes após merge:
0

Resumo da diferença absoluta percentual:


count    1660.000000
mean        0.219038
std         1.864158
min         0.000001
25%         0.001417
50%         0.003928
75%         0.013195
max        42.885356
Name: diferenca_area_abs_pct, dtype: float64

In [83]:
# ============================================================
# MAIORES DIFERENÇAS — MAPBIOMAS × IBGE
# ============================================================

maiores_diferencas_area = (
    comparacao_area_ibge
    .sort_values(
        "diferenca_area_abs_pct",
        ascending=False
    )
    [
        [
            "codigo_ibge_final",
            "municipio_ibge_final",
            "uf_ibge_final",
            "area_total_media_ha",
            "area_ibge_ha",
            "diferenca_area_ha",
            "diferenca_area_pct",
            "diferenca_area_abs_pct"
        ]
    ]
)


display(
    maiores_diferencas_area.head(30)
)

,codigo_ibge_final,municipio_ibge_final,uf_ibge_final,area_total_media_ha,area_ibge_ha,diferenca_area_ha,diferenca_area_pct,diferenca_area_abs_pct
1351,5106240,Nova Ubiratã,MT,1.246071e+06,872077.7,373993.626379,42.885356,42.885356
486,4205407,Florianópolis,SC,4.389180e+04,67484.4,-23592.599565,-34.960079,34.960079
495,4206009,Governador Celso Ramos,SC,9.374099e+03,12755.6,-3381.501223,-26.509935,26.509935
647,4216602,São José,SC,1.143697e+04,15049.9,-3612.929156,-24.006333,24.006333
1406,5108402,Várzea Grande,MT,7.242764e+04,93988.7,-21561.064760,-22.940061,22.940061
579,4211900,Palhoça,SC,3.276795e+04,39485.0,-6717.048421,-17.011646,17.011646
434,4202305,Biguaçu,SC,3.166517e+04,36575.5,-4910.328537,-13.425185,13.425185
1397,5107925,Sorriso,MT,9.293578e+05,845483.0,83874.773414,9.920338,9.920338
1124,4320453,Sério,RS,9.813751e+03,10542.8,-729.049472,-6.915141,6.915141
1103,4319356,São Pedro da Serra,RS,3.521263e+03,3763.1,-241.836822,-6.426532,6.426532


In [84]:
print(
    "Quantidade com diferença absoluta <= 0,1%:"
)

print(
    (
        comparacao_area_ibge[
            "diferenca_area_abs_pct"
        ] <= 0.1
    ).sum()
)


print(
    "\nQuantidade com diferença absoluta <= 1%:"
)

print(
    (
        comparacao_area_ibge[
            "diferenca_area_abs_pct"
        ] <= 1
    ).sum()
)


print(
    "\nQuantidade com diferença absoluta > 1%:"
)

print(
    (
        comparacao_area_ibge[
            "diferenca_area_abs_pct"
        ] > 1
    ).sum()
)

Quantidade com diferença absoluta <= 0,1%:
1485

Quantidade com diferença absoluta <= 1%:
1607

Quantidade com diferença absoluta > 1%:
53


In [85]:
# ============================================================
# DIAGNÓSTICO TERRITORIAL — BOA ESPERANÇA DO NORTE
# ============================================================

MUNICIPIOS_RELACIONADOS_BEN = [
    "Nova Ubiratã",
    "Sorriso",
    "Paranatinga",
    "Vera"
]


diagnostico_boa_esperanca_norte = (
    comparacao_area_ibge[
        comparacao_area_ibge[
            "municipio_ibge_final"
        ].isin(
            MUNICIPIOS_RELACIONADOS_BEN
        )
    ][
        [
            "codigo_ibge_final",
            "municipio_ibge_final",
            "uf_ibge_final",
            "area_total_media_ha",
            "area_ibge_ha",
            "diferenca_area_ha",
            "diferenca_area_pct"
        ]
    ]
    .sort_values(
        "municipio_ibge_final"
    )
)


display(
    diagnostico_boa_esperanca_norte
)


print(
    "\nSoma das diferenças de área dos quatro municípios:"
)

print(
    diagnostico_boa_esperanca_norte[
        "diferenca_area_ha"
    ].sum()
)

,codigo_ibge_final,municipio_ibge_final,uf_ibge_final,area_total_media_ha,area_ibge_ha,diferenca_area_ha,diferenca_area_pct
1351,5106240,Nova Ubiratã,MT,1.246071e+06,872077.7,373993.626379,42.885356
1357,5106307,Paranatinga,MT,2.416656e+06,2412742.9,3913.344566,0.162195
1397,5107925,Sorriso,MT,9.293578e+05,845483.0,83874.773414,9.920338
1407,5108501,Vera,MT,3.058394e+05,297867.4,7971.963969,2.676347



Soma das diferenças de área dos quatro municípios:
469753.7083272786


In [92]:
# ============================================================
# LISTAGEM DOS CASOS COM DIFERENÇA > 1%
# ============================================================

casos_acima_1pct_ordenados = (
    casos_acima_1pct
    .sort_values(
        "diferenca_area_abs_pct",
        ascending=False
    )
    [
        [
            "codigo_ibge_final",
            "municipio_ibge_final",
            "uf_ibge_final",
            "area_total_media_ha",
            "area_ibge_ha",
            "diferenca_area_ha",
            "diferenca_area_pct",
            "diferenca_area_abs_pct",
            "sentido_diferenca"
        ]
    ]
)

display(
    casos_acima_1pct_ordenados
)

,codigo_ibge_final,municipio_ibge_final,uf_ibge_final,area_total_media_ha,area_ibge_ha,diferenca_area_ha,diferenca_area_pct,diferenca_area_abs_pct,sentido_diferenca
1351,5106240,Nova Ubiratã,MT,1.246071e+06,872077.7,373993.626379,42.885356,42.885356,MapBiomas_maior
486,4205407,Florianópolis,SC,4.389180e+04,67484.4,-23592.599565,-34.960079,34.960079,MapBiomas_menor
495,4206009,Governador Celso Ramos,SC,9.374099e+03,12755.6,-3381.501223,-26.509935,26.509935,MapBiomas_menor
647,4216602,São José,SC,1.143697e+04,15049.9,-3612.929156,-24.006333,24.006333,MapBiomas_menor
1406,5108402,Várzea Grande,MT,7.242764e+04,93988.7,-21561.064760,-22.940061,22.940061,MapBiomas_menor
579,4211900,Palhoça,SC,3.276795e+04,39485.0,-6717.048421,-17.011646,17.011646,MapBiomas_menor
434,4202305,Biguaçu,SC,3.166517e+04,36575.5,-4910.328537,-13.425185,13.425185,MapBiomas_menor
1397,5107925,Sorriso,MT,9.293578e+05,845483.0,83874.773414,9.920338,9.920338,MapBiomas_maior
1124,4320453,Sério,RS,9.813751e+03,10542.8,-729.049472,-6.915141,6.915141,MapBiomas_menor
1103,4319356,São Pedro da Serra,RS,3.521263e+03,3763.1,-241.836822,-6.426532,6.426532,MapBiomas_menor


In [94]:
# ============================================================
# SALVAR AUDITORIA MAPBIOMAS × IBGE
# ============================================================

ARQUIVO_AUDITORIA_AREA_IBGE = (
    QUALITY_MAPBIOMAS_COBERTURA_DIR
    / "auditoria_area_mapbiomas_vs_ibge_2024.csv"
)

comparacao_area_ibge.to_csv(
    ARQUIVO_AUDITORIA_AREA_IBGE,
    index=False,
    encoding="utf-8-sig"
)


ARQUIVO_CASOS_ACIMA_1PCT = (
    QUALITY_MAPBIOMAS_COBERTURA_DIR
    / "auditoria_area_casos_diferenca_maior_1pct.csv"
)

casos_acima_1pct_ordenados.to_csv(
    ARQUIVO_CASOS_ACIMA_1PCT,
    index=False,
    encoding="utf-8-sig"
)


print(
    "Auditoria completa:"
)

print(
    ARQUIVO_AUDITORIA_AREA_IBGE
)


print(
    "\nCasos com diferença > 1%:"
)

print(
    ARQUIVO_CASOS_ACIMA_1PCT
)

Auditoria completa:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\databases_processed\mapbiomas_cobertura\quality\auditoria_area_mapbiomas_vs_ibge_2024.csv

Casos com diferença > 1%:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\databases_processed\mapbiomas_cobertura\quality\auditoria_area_casos_diferenca_maior_1pct.csv


In [95]:
# ============================================================
# VALIDAÇÃO — FECHAMENTO DA ÁREA TOTAL MAPEADA
# ============================================================

COLUNAS_PARTICAO_TERRITORIAL = [
    "area_floresta_ha",
    "area_formacao_natural_nao_florestal_ha",
    "area_agropecuaria_ha",
    "area_nao_vegetada_ha",
    "area_agua_natural_ha",
    "area_aquicultura_ha",
    "area_nao_observada_ha"
]


validacao_particao = (
    mapbiomas_municipio_ano
    .copy()
)


validacao_particao[
    "soma_particao_territorial_ha"
] = (
    validacao_particao[
        COLUNAS_PARTICAO_TERRITORIAL
    ]
    .sum(axis=1)
)


validacao_particao[
    "diferenca_particao_ha"
] = (
    validacao_particao[
        "area_total_mapeada_ha"
    ]
    -
    validacao_particao[
        "soma_particao_territorial_ha"
    ]
)


print(
    "Maior diferença absoluta:"
)

print(
    validacao_particao[
        "diferenca_particao_ha"
    ]
    .abs()
    .max()
)


print(
    "\nRegistros com diferença absoluta > 0,000001 ha:"
)

print(
    (
        validacao_particao[
            "diferenca_particao_ha"
        ]
        .abs()
        > 1e-6
    ).sum()
)

Maior diferença absoluta:
1.862645149230957e-09

Registros com diferença absoluta > 0,000001 ha:
0


In [96]:
# ============================================================
# VALIDAÇÃO — HIERARQUIA DOS INDICADORES
# ============================================================

TOLERANCIA_AREA = 1e-6


soma_componentes_agropecuaria = (
    mapbiomas_municipio_ano[
        "area_pastagem_ha"
    ]
    +
    mapbiomas_municipio_ano[
        "area_agricultura_ha"
    ]
    +
    mapbiomas_municipio_ano[
        "area_silvicultura_ha"
    ]
    +
    mapbiomas_municipio_ano[
        "area_mosaico_usos_ha"
    ]
)


validacoes_hierarquia = {
    
    "agropecuaria_diferente_componentes": (
        (
            mapbiomas_municipio_ano[
                "area_agropecuaria_ha"
            ]
            -
            soma_componentes_agropecuaria
        )
        .abs()
        > TOLERANCIA_AREA
    ).sum(),

    "agricultura_maior_agropecuaria": (
        mapbiomas_municipio_ano[
            "area_agricultura_ha"
        ]
        >
        mapbiomas_municipio_ano[
            "area_agropecuaria_ha"
        ]
        + TOLERANCIA_AREA
    ).sum(),

    "lavoura_temporaria_maior_agricultura": (
        mapbiomas_municipio_ano[
            "area_lavoura_temporaria_ha"
        ]
        >
        mapbiomas_municipio_ano[
            "area_agricultura_ha"
        ]
        + TOLERANCIA_AREA
    ).sum(),

    "soja_maior_lavoura_temporaria": (
        mapbiomas_municipio_ano[
            "area_soja_mapbiomas_ha"
        ]
        >
        mapbiomas_municipio_ano[
            "area_lavoura_temporaria_ha"
        ]
        + TOLERANCIA_AREA
    ).sum(),

    "pastagem_maior_agropecuaria": (
        mapbiomas_municipio_ano[
            "area_pastagem_ha"
        ]
        >
        mapbiomas_municipio_ano[
            "area_agropecuaria_ha"
        ]
        + TOLERANCIA_AREA
    ).sum()
}


display(
    pd.Series(
        validacoes_hierarquia,
        name="quantidade_inconsistencias"
    )
)

agropecuaria_diferente_componentes      0
agricultura_maior_agropecuaria          0
lavoura_temporaria_maior_agricultura    0
soja_maior_lavoura_temporaria           0
pastagem_maior_agropecuaria             0
Name: quantidade_inconsistencias, dtype: int64

In [97]:
# ============================================================
# INCORPORAR CONTROLE TERRITORIAL IBGE
# ============================================================

auditoria_area_merge = (
    comparacao_area_ibge[
        [
            "codigo_ibge_final",
            "area_ibge_ha",
            "diferenca_area_pct",
            "diferenca_area_abs_pct"
        ]
    ]
    .copy()
)


mapbiomas_municipio_ano_auditado = (
    mapbiomas_municipio_ano
    .merge(
        auditoria_area_merge,
        on="codigo_ibge_final",
        how="left",
        validate="many_to_one"
    )
)


mapbiomas_municipio_ano_auditado[
    "faixa_diferenca_area_ibge"
] = np.select(
    [
        mapbiomas_municipio_ano_auditado[
            "diferenca_area_abs_pct"
        ] <= 0.1,

        mapbiomas_municipio_ano_auditado[
            "diferenca_area_abs_pct"
        ] <= 1
    ],
    [
        "ate_0_1_pct",
        "entre_0_1_e_1_pct"
    ],
    default="acima_1_pct"
)


print(
    "Dimensão:"
)

print(
    mapbiomas_municipio_ano_auditado.shape
)


print(
    "\nFaixas de diferença territorial:"
)

display(
    mapbiomas_municipio_ano_auditado[
        [
            "codigo_ibge_final",
            "faixa_diferenca_area_ibge"
        ]
    ]
    .drop_duplicates()[
        "faixa_diferenca_area_ibge"
    ]
    .value_counts()
)

Dimensão:
(9960, 25)

Faixas de diferença territorial:


faixa_diferenca_area_ibge
ate_0_1_pct          1485
entre_0_1_e_1_pct     122
acima_1_pct            53
Name: count, dtype: int64

In [98]:
# ============================================================
# CONSTRUÇÃO DA BASE CURATED
# ============================================================

mapbiomas_curated = (
    mapbiomas_municipio_ano_auditado
    .copy()
    .rename(
        columns={
            "codigo_ibge_final": "codigo_ibge",
            "municipio_ibge_final": "municipio",
            "uf_ibge_final": "uf"
        }
    )
)


REGIAO_POR_UF = {
    "DF": "Centro-Oeste",
    "GO": "Centro-Oeste",
    "MS": "Centro-Oeste",
    "MT": "Centro-Oeste",
    "PR": "Sul",
    "RS": "Sul",
    "SC": "Sul"
}


mapbiomas_curated[
    "regiao"
] = (
    mapbiomas_curated[
        "uf"
    ]
    .map(
        REGIAO_POR_UF
    )
)


mapbiomas_curated[
    "codigo_ibge"
] = (
    mapbiomas_curated[
        "codigo_ibge"
    ]
    .astype("string")
    .str.zfill(7)
)


print(
    "Dimensão:"
)

print(
    mapbiomas_curated.shape
)


print(
    "\nRegiões:"
)

display(
    mapbiomas_curated[
        "regiao"
    ]
    .value_counts()
)

Dimensão:
(9960, 26)

Regiões:


regiao
Sul             7158
Centro-Oeste    2802
Name: count, dtype: int64

In [99]:
# ============================================================
# INDICADORES PERCENTUAIS
# ============================================================

MAPA_PERCENTUAIS = {
    "area_floresta_ha": "pct_floresta",
    "area_formacao_natural_nao_florestal_ha": "pct_formacao_natural_nao_florestal",
    "area_cobertura_natural_ha": "pct_cobertura_natural",
    "area_agropecuaria_ha": "pct_agropecuaria",
    "area_pastagem_ha": "pct_pastagem",
    "area_agricultura_ha": "pct_agricultura",
    "area_lavoura_temporaria_ha": "pct_lavoura_temporaria",
    "area_soja_mapbiomas_ha": "pct_soja_mapbiomas",
    "area_silvicultura_ha": "pct_silvicultura",
    "area_mosaico_usos_ha": "pct_mosaico_usos",
    "area_nao_vegetada_ha": "pct_area_nao_vegetada",
    "area_agua_natural_ha": "pct_agua_natural",
    "area_aquicultura_ha": "pct_aquicultura",
    "area_nao_observada_ha": "pct_nao_observada"
}


for coluna_area, coluna_pct in MAPA_PERCENTUAIS.items():

    mapbiomas_curated[
        coluna_pct
    ] = (
        mapbiomas_curated[
            coluna_area
        ]
        /
        mapbiomas_curated[
            "area_total_mapeada_ha"
        ]
        * 100
    )

In [100]:
COLUNAS_PCT_PARTICAO = [
    "pct_floresta",
    "pct_formacao_natural_nao_florestal",
    "pct_agropecuaria",
    "pct_area_nao_vegetada",
    "pct_agua_natural",
    "pct_aquicultura",
    "pct_nao_observada"
]


mapbiomas_curated[
    "pct_fechamento_territorial"
] = (
    mapbiomas_curated[
        COLUNAS_PCT_PARTICAO
    ]
    .sum(axis=1)
)

In [101]:
# ============================================================
# ORGANIZAÇÃO DA CURATED
# ============================================================

COLUNAS_CURATED = [
    # Identificação
    "codigo_ibge",
    "municipio",
    "uf",
    "regiao",
    "ano",

    # Biomas
    "quantidade_biomas",
    "biomas_presentes",

    # Referência territorial
    "area_total_mapeada_ha",
    "area_ibge_ha",
    "diferenca_area_pct",
    "diferenca_area_abs_pct",
    "faixa_diferenca_area_ibge",

    # Cobertura natural
    "area_floresta_ha",
    "area_formacao_natural_nao_florestal_ha",
    "area_cobertura_natural_ha",

    # Agropecuária
    "area_agropecuaria_ha",
    "area_pastagem_ha",
    "area_agricultura_ha",
    "area_lavoura_temporaria_ha",
    "area_soja_mapbiomas_ha",
    "area_silvicultura_ha",
    "area_mosaico_usos_ha",

    # Outras classes
    "area_nao_vegetada_ha",
    "area_agua_natural_ha",
    "area_aquicultura_ha",
    "area_nao_observada_ha",

    # Percentuais
    "pct_floresta",
    "pct_formacao_natural_nao_florestal",
    "pct_cobertura_natural",
    "pct_agropecuaria",
    "pct_pastagem",
    "pct_agricultura",
    "pct_lavoura_temporaria",
    "pct_soja_mapbiomas",
    "pct_silvicultura",
    "pct_mosaico_usos",
    "pct_area_nao_vegetada",
    "pct_agua_natural",
    "pct_aquicultura",
    "pct_nao_observada",

    # Controle
    "pct_fechamento_territorial"
]


mapbiomas_curated = (
    mapbiomas_curated[
        COLUNAS_CURATED
    ]
    .sort_values(
        [
            "codigo_ibge",
            "ano"
        ]
    )
    .reset_index(
        drop=True
    )
)

In [102]:
print(
    "Dimensão final:"
)

print(
    mapbiomas_curated.shape
)


print(
    "\nDuplicatas codigo_ibge + ano:"
)

print(
    mapbiomas_curated
    .duplicated(
        subset=[
            "codigo_ibge",
            "ano"
        ]
    )
    .sum()
)


print(
    "\nValores ausentes:"
)

print(
    mapbiomas_curated
    .isna()
    .sum()
    .sum()
)


print(
    "\nFaixa do fechamento territorial percentual:"
)

print(
    mapbiomas_curated[
        "pct_fechamento_territorial"
    ].min()
)

print(
    mapbiomas_curated[
        "pct_fechamento_territorial"
    ].max()
)

Dimensão final:
(9960, 41)

Duplicatas codigo_ibge + ano:
0

Valores ausentes:
0

Faixa do fechamento territorial percentual:
99.99999999999997
100.00000000000004


In [103]:
COLUNAS_PERCENTUAIS = list(
    MAPA_PERCENTUAIS.values()
)


percentuais_fora_faixa = 0


for coluna in COLUNAS_PERCENTUAIS:

    quantidade = (
        (
            mapbiomas_curated[
                coluna
            ] < -1e-9
        )
        |
        (
            mapbiomas_curated[
                coluna
            ] > 100 + 1e-9
        )
    ).sum()

    percentuais_fora_faixa += quantidade

    print(
        coluna,
        "->",
        quantidade
    )


print(
    "\nTotal de percentuais fora de 0–100:"
)

print(
    percentuais_fora_faixa
)

pct_floresta -> 0
pct_formacao_natural_nao_florestal -> 0
pct_cobertura_natural -> 0
pct_agropecuaria -> 0
pct_pastagem -> 0
pct_agricultura -> 0
pct_lavoura_temporaria -> 0
pct_soja_mapbiomas -> 0
pct_silvicultura -> 0
pct_mosaico_usos -> 0
pct_area_nao_vegetada -> 0
pct_agua_natural -> 0
pct_aquicultura -> 0
pct_nao_observada -> 0

Total de percentuais fora de 0–100:
0


In [104]:
# ============================================================
# EXPORTAÇÃO — CURATED MAPBIOMAS COBERTURA
# ============================================================

ARQUIVO_CURATED_MAPBIOMAS_COBERTURA = (
    CURATED_MAPBIOMAS_COBERTURA_DIR
    / "mapbiomas_cobertura_municipio_ano_2019_2024.csv"
)


mapbiomas_curated.to_csv(
    ARQUIVO_CURATED_MAPBIOMAS_COBERTURA,
    index=False,
    encoding="utf-8-sig"
)


print(
    "Curated salva com sucesso:"
)

print(
    ARQUIVO_CURATED_MAPBIOMAS_COBERTURA
)


print(
    "\nDimensão:"
)

print(
    mapbiomas_curated.shape
)


print(
    "\nPrimeiros registros:"
)

display(
    mapbiomas_curated.head()
)

Curated salva com sucesso:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\databases_curated\mapbiomas_cobertura\mapbiomas_cobertura_municipio_ano_2019_2024.csv

Dimensão:
(9960, 41)

Primeiros registros:


,codigo_ibge,municipio,uf,regiao,ano,quantidade_biomas,biomas_presentes,area_total_mapeada_ha,area_ibge_ha,diferenca_area_pct,diferenca_area_abs_pct,faixa_diferenca_area_ibge,area_floresta_ha,area_formacao_natural_nao_florestal_ha,area_cobertura_natural_ha,area_agropecuaria_ha,area_pastagem_ha,area_agricultura_ha,area_lavoura_temporaria_ha,area_soja_mapbiomas_ha,area_silvicultura_ha,area_mosaico_usos_ha,area_nao_vegetada_ha,area_agua_natural_ha,area_aquicultura_ha,area_nao_observada_ha,pct_floresta,pct_formacao_natural_nao_florestal,pct_cobertura_natural,pct_agropecuaria,pct_pastagem,pct_agricultura,pct_lavoura_temporaria,pct_soja_mapbiomas,pct_silvicultura,pct_mosaico_usos,pct_area_nao_vegetada,pct_agua_natural,pct_aquicultura,pct_nao_observada,pct_fechamento_territorial
0,4100103,Abatiá,PR,Sul,2019,1,Mata Atlântica,22872.050344,22871.7,0.001532,0.001532,ate_0_1_pct,2110.560484,20.153876,2130.714359,20501.078533,5773.710590,8494.500847,8183.880715,6337.048921,155.260906,6077.606191,159.532255,80.725196,0.0,0.0,9.227684,0.088116,9.315800,89.633759,25.243520,37.139219,35.781142,27.706519,0.678824,26.572197,0.697499,0.352943,0.0,0.0,100.0
1,4100103,Abatiá,PR,Sul,2020,1,Mata Atlântica,22872.050344,22871.7,0.001532,0.001532,ate_0_1_pct,2138.038478,24.102157,2162.140635,20465.703400,5515.560753,8557.179526,8245.983106,6814.108692,153.615340,6239.347781,163.892775,80.313534,0.0,0.0,9.347822,0.105378,9.453200,89.479094,24.114851,37.413259,36.052662,29.792295,0.671629,27.279355,0.716564,0.351143,0.0,0.0,100.0
2,4100103,Abatiá,PR,Sul,2021,1,Mata Atlântica,22872.050344,22871.7,0.001532,0.001532,ate_0_1_pct,2144.043562,24.842192,2168.885754,20457.066441,5136.433064,8573.384341,8262.928519,6855.572934,153.039484,6594.209551,165.620277,80.477873,0.0,0.0,9.374077,0.108614,9.482691,89.441332,22.457248,37.484109,36.126750,29.973583,0.669111,28.830863,0.724116,0.351861,0.0,0.0,100.0
3,4100103,Abatiá,PR,Sul,2022,1,Mata Atlântica,22872.050344,22871.7,0.001532,0.001532,ate_0_1_pct,2120.430937,27.228733,2147.659670,20463.566371,4904.415611,8601.355462,8291.722262,6969.448836,152.628175,6805.167123,176.562297,84.262006,0.0,0.0,9.270839,0.119048,9.389887,89.469750,21.442833,37.606403,36.252641,30.471465,0.667313,29.753201,0.771957,0.368406,0.0,0.0,100.0
4,4100103,Abatiá,PR,Sul,2023,1,Mata Atlântica,22872.050344,22871.7,0.001532,0.001532,ate_0_1_pct,2111.213707,26.652856,2137.866563,20467.189193,4644.000071,8617.563791,8307.766021,7127.089229,152.134601,7053.490730,180.100145,86.894442,0.0,0.0,9.230540,0.116530,9.347070,89.485590,20.304258,37.677268,36.322787,31.160692,0.665155,30.838909,0.787425,0.379915,0.0,0.0,100.0


In [105]:
# ============================================================
# VALIDAÇÃO FINAL — RELEITURA DO ARQUIVO CURATED
# ============================================================

mapbiomas_curated_check = pd.read_csv(
    ARQUIVO_CURATED_MAPBIOMAS_COBERTURA,
    dtype={
        "codigo_ibge": "string"
    }
)


mapbiomas_curated_check[
    "codigo_ibge"
] = (
    mapbiomas_curated_check[
        "codigo_ibge"
    ]
    .str.zfill(7)
)


print(
    "Dimensão do arquivo relido:"
)

print(
    mapbiomas_curated_check.shape
)


print(
    "\nDuplicatas codigo_ibge + ano:"
)

print(
    mapbiomas_curated_check
    .duplicated(
        subset=[
            "codigo_ibge",
            "ano"
        ]
    )
    .sum()
)


print(
    "\nValores ausentes:"
)

print(
    mapbiomas_curated_check
    .isna()
    .sum()
    .sum()
)


print(
    "\nUnidades territoriais:"
)

print(
    mapbiomas_curated_check[
        "codigo_ibge"
    ]
    .nunique()
)


print(
    "\nAnos:"
)

print(
    sorted(
        mapbiomas_curated_check[
            "ano"
        ]
        .unique()
        .tolist()
    )
)

Dimensão do arquivo relido:
(9960, 41)

Duplicatas codigo_ibge + ano:
0

Valores ausentes:
0

Unidades territoriais:
1660

Anos:
[2019, 2020, 2021, 2022, 2023, 2024]
